In [1]:
import os
import glob
import warnings

import numpy as np
import pandas as pd

from IPython.display import display

from sklearn.metrics import average_precision_score
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression, Ridge, ElasticNet

warnings.filterwarnings("ignore")

In [2]:
# ============================================================
# Step 1. Load Data
# Retroviral Wall Challenge Final Pipeline
# ============================================================

# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------
COMP_DIR = "/kaggle/input/competitions/retroviral-challenge-predict"

print("Competition directory exists:", os.path.exists(COMP_DIR))
print("Available files:")
for path in sorted(glob.glob(os.path.join(COMP_DIR, "*"))):
    print(" -", os.path.basename(path))


# ------------------------------------------------------------
# 2. Robust data loader
# ------------------------------------------------------------
def load_retroviral_data(data_dir=COMP_DIR):
    """
    Load Retroviral Wall Challenge data.

    Preferred format:
        train.csv
        test.csv

    Fallback format:
        rt_sequences.csv
        handcrafted_features.csv
        esm2_embeddings.npz
        structures/

    Returns
    -------
    train_df : pd.DataFrame
        Training dataframe with labels.
    test_df : pd.DataFrame
        Test/new dataframe without labels if available.
    meta : dict
        Metadata including available files and paths.
    """

    meta = {
        "data_dir": data_dir,
        "used_format": None,
        "available_files": sorted(os.listdir(data_dir)) if os.path.exists(data_dir) else [],
    }

    train_path = os.path.join(data_dir, "train.csv")
    test_path = os.path.join(data_dir, "test.csv")

    # --------------------------------------------------------
    # Case A: competition-provided train.csv / test.csv
    # --------------------------------------------------------
    if os.path.exists(train_path):
        train_df = pd.read_csv(train_path)
        meta["used_format"] = "train_test_csv"

        if os.path.exists(test_path):
            test_df = pd.read_csv(test_path)
        else:
            # Some Phase 1 notebooks only need train OOF.
            test_df = None

        return train_df, test_df, meta

    # --------------------------------------------------------
    # Case B: separated files from official description
    # --------------------------------------------------------
    seq_path = os.path.join(data_dir, "rt_sequences.csv")
    feat_path = os.path.join(data_dir, "handcrafted_features.csv")

    if os.path.exists(seq_path) and os.path.exists(feat_path):
        seq_df = pd.read_csv(seq_path)
        feat_df = pd.read_csv(feat_path)

        # Try common key names
        possible_keys = ["rt_name", "id", "name"]
        merge_key = None

        for key in possible_keys:
            if key in seq_df.columns and key in feat_df.columns:
                merge_key = key
                break

        if merge_key is None:
            raise ValueError(
                "Could not find a common merge key between rt_sequences.csv "
                "and handcrafted_features.csv."
            )

        train_df = seq_df.merge(feat_df, on=merge_key, how="left")
        test_df = None
        meta["used_format"] = "sequences_plus_handcrafted"
        meta["merge_key"] = merge_key

        return train_df, test_df, meta

    raise FileNotFoundError(
        "Could not find train.csv/test.csv or rt_sequences.csv + handcrafted_features.csv."
    )


train_df, test_df, meta = load_retroviral_data(COMP_DIR)


# ------------------------------------------------------------
# 3. Basic checks
# ------------------------------------------------------------
print("\n" + "=" * 80)
print("LOAD SUMMARY")
print("=" * 80)
print("Used format:", meta["used_format"])
print("Train shape:", train_df.shape)

if test_df is not None:
    print("Test shape:", test_df.shape)
else:
    print("Test shape: None")

print("\nTrain columns:")
print(train_df.columns.tolist())

if test_df is not None:
    print("\nTest columns:")
    print(test_df.columns.tolist())


# ------------------------------------------------------------
# 4. Standard column names
# ------------------------------------------------------------
ID_COL = "rt_name"
FAMILY_COL = "rt_family"
TARGET_COL = "active"
EFF_COL = "pe_efficiency_pct"

required_train_cols = [ID_COL, FAMILY_COL, TARGET_COL, EFF_COL]
missing_cols = [c for c in required_train_cols if c not in train_df.columns]

if missing_cols:
    raise ValueError(f"Missing required train columns: {missing_cols}")

print("\n" + "=" * 80)
print("TARGET / FAMILY CHECK")
print("=" * 80)

display(train_df[[ID_COL, FAMILY_COL, TARGET_COL, EFF_COL]].head())

print("\nFamily breakdown:")
family_summary = (
    train_df
    .groupby(FAMILY_COL)
    .agg(
        n=(ID_COL, "count"),
        active=(TARGET_COL, "sum"),
        inactive=(TARGET_COL, lambda x: int((x == 0).sum())),
        mean_efficiency=(EFF_COL, "mean"),
        max_efficiency=(EFF_COL, "max"),
    )
    .reset_index()
    .sort_values("n", ascending=False)
)

display(family_summary)

print("\nActive distribution:")
display(train_df[TARGET_COL].value_counts().rename("count"))

print("\nEfficiency summary:")
display(train_df[EFF_COL].describe())


# ------------------------------------------------------------
# 5. Numeric feature check
# ------------------------------------------------------------
non_feature_cols = [
    ID_COL,
    FAMILY_COL,
    TARGET_COL,
    EFF_COL,
    "sequence",
    "yxdd_seq",
]

numeric_feature_cols = [
    c for c in train_df.columns
    if c not in non_feature_cols
    and pd.api.types.is_numeric_dtype(train_df[c])
]

print("\n" + "=" * 80)
print("FEATURE CHECK")
print("=" * 80)
print("Number of numeric feature columns:", len(numeric_feature_cols))
print(numeric_feature_cols)

missing_rate = (
    train_df[numeric_feature_cols]
    .isna()
    .mean()
    .sort_values(ascending=False)
)

print("\nTop missing-rate features:")
display(missing_rate.head(20))


# ------------------------------------------------------------
# 6. Data objects for later cells
# ------------------------------------------------------------
y_active = train_df[TARGET_COL].astype(int).values
y_eff = train_df[EFF_COL].astype(float).values
families = train_df[FAMILY_COL].values
rt_names = train_df[ID_COL].values

print("\nReady.")
print("n_train:", len(train_df))
print("n_active:", int(y_active.sum()))
print("n_inactive:", int((y_active == 0).sum()))
print("n_families:", train_df[FAMILY_COL].nunique())

# ------------------------------------------------------------
# 7. Compatibility aliases for downstream cells
# ------------------------------------------------------------
train = train_df.copy()
test = test_df.copy() if test_df is not None else None

id_col = ID_COL
family_col = FAMILY_COL
group_col = FAMILY_COL
target_col = TARGET_COL
eff_col = EFF_COL

y = y_active.copy()
eff = y_eff.copy()
groups = families.copy()

print("\nCompatibility aliases ready.")

Competition directory exists: True
Available files:
 - esm2_embeddings.npz
 - family_splits.csv
 - feature_dictionary.csv
 - sample_submission.csv
 - structures
 - test.csv
 - train.csv

LOAD SUMMARY
Used format: train_test_csv
Train shape: (57, 71)
Test shape: (57, 69)

Train columns:
['rt_name', 'sequence', 'active', 'pe_efficiency_pct', 'protein_length_aa', 'perplexity', 'instability_index', 'gravy', 't40_raw', 't45_raw', 't50_raw', 't55_raw', 't60_raw', 't65_raw', 't70_raw', 't75_raw', 't80_raw', 'thermophilicity_num', 'triad_found_bin', 'D1_D2_dist', 'D2_D3_dist', 'triad_best_rmsd', 'hydrophobic_per_res', 'salt_per_res', 'hbonds_per_res', 'pocket_hydrophobic_per_res', 'pocket_hbonds_per_res', 'whole_mean_hydrophobicity', 'whole_std_hydrophobicity', 'yxdd_hydrophobic_fraction', 'yxdd_mean_hydrophobicity', 'hairpin_pass', 'yxdd_y_is_strand', 'yxdd_x_is_turn', 'yxdd_d1_is_turn', 'yxdd_d2_is_strand', 'foldseek_best_TM', 'foldseek_best_LDDT', 'foldseek_best_fident', 'foldseek_TM_HIV1',

,rt_name,rt_family,active,pe_efficiency_pct
0,A.platensis-Cas1-RT,CRISPR-associated,0,0.0
1,CRISPRRT2-RT,CRISPR-associated,0,0.0
2,FuRT-Cas1-RT,CRISPR-associated,0,0.0
3,Med-CasRT,CRISPR-associated,0,0.0
4,ROSE-CRISPR-RT,CRISPR-associated,0,0.0



Family breakdown:


,rt_family,n,active,inactive,mean_efficiency,max_efficiency
5,Retroviral,18,12,6,10.694444,41.0
4,Retron,12,5,7,1.166667,8.0
2,LTR_Retrotransposon,11,2,9,3.909091,34.0
1,Group_II_Intron,5,2,3,2.700000,12.5
0,CRISPR-associated,5,0,5,0.000000,0.0
3,Other,5,0,5,0.000000,0.0
6,Unclassified,1,0,1,0.000000,0.0



Active distribution:


...


Efficiency summary:


count    57.000000
mean      4.614035
std       9.362262
min       0.000000
25%       0.000000
50%       0.000000
75%       3.500000
max      41.000000
Name: pe_efficiency_pct, dtype: float64


FEATURE CHECK
Number of numeric feature columns: 65
['protein_length_aa', 'perplexity', 'instability_index', 'gravy', 't40_raw', 't45_raw', 't50_raw', 't55_raw', 't60_raw', 't65_raw', 't70_raw', 't75_raw', 't80_raw', 'thermophilicity_num', 'triad_found_bin', 'D1_D2_dist', 'D2_D3_dist', 'triad_best_rmsd', 'hydrophobic_per_res', 'salt_per_res', 'hbonds_per_res', 'pocket_hydrophobic_per_res', 'pocket_hbonds_per_res', 'whole_mean_hydrophobicity', 'whole_std_hydrophobicity', 'yxdd_hydrophobic_fraction', 'yxdd_mean_hydrophobicity', 'hairpin_pass', 'yxdd_y_is_strand', 'yxdd_x_is_turn', 'yxdd_d1_is_turn', 'yxdd_d2_is_strand', 'foldseek_best_TM', 'foldseek_best_LDDT', 'foldseek_best_fident', 'foldseek_TM_HIV1', 'foldseek_TM_MMLV', 'foldseek_TM_MMLVPE', 'foldseek_TM_Retron', 'foldseek_TM_LTRRetrotransposon', 'foldseek_TM_Group2Intron', 'foldseek_TM_Telomerase', 'seq_length', 'log_likelihood', 'net_charge', 'camsol', 'helix_beta_ratio', 'yxdd_std_hydrophobicity', 'n_residues', 'sasa_per_res', 'a

connection_mean_pot          0.578947
yxdd_mean_hydrophobicity     0.122807
D2_D3_dist                   0.122807
yxdd_hydrophobic_fraction    0.122807
D1_D2_dist                   0.122807
yxdd_std_hydrophobicity      0.122807
yxdd_5A_mean_pot             0.122807
triad_best_rmsd              0.122807
hairpin_confidence           0.070175
yxdd_d1_is_turn              0.017544
yxdd_y_is_strand             0.017544
yxdd_d2_is_strand            0.017544
thumb_mean_pot               0.017544
yxdd_x_is_turn               0.017544
protein_length_aa            0.000000
perplexity                   0.000000
t60_raw                      0.000000
thermophilicity_num          0.000000
triad_found_bin              0.000000
t75_raw                      0.000000
dtype: float64


Ready.
n_train: 57
n_active: 21
n_inactive: 36
n_families: 7

Compatibility aliases ready.


In [3]:
# ============================================================
# Step 2. Metrics + LOFO Helpers
# Retroviral Wall Challenge Final Pipeline
# ============================================================
# ------------------------------------------------------------
# 1. Official-style metric functions
# ------------------------------------------------------------
def weighted_spearman(pred, true_eff, weights):
    """
    Weighted Spearman correlation.

    Parameters
    ----------
    pred : array-like
        Predicted continuous scores.
    true_eff : array-like
        True PE efficiency percentages.
    weights : array-like
        Usually pe_efficiency_pct + epsilon.

    Returns
    -------
    float
        Weighted Spearman correlation, floored at 0.
    """
    pred = np.asarray(pred, dtype=float)
    true_eff = np.asarray(true_eff, dtype=float)
    weights = np.asarray(weights, dtype=float)

    pred_ranks = np.argsort(np.argsort(pred)).astype(float)
    true_ranks = np.argsort(np.argsort(true_eff)).astype(float)

    w = weights / weights.sum()

    mu_p = np.dot(w, pred_ranks)
    mu_t = np.dot(w, true_ranks)

    cov = np.sum(w * (pred_ranks - mu_p) * (true_ranks - mu_t))
    std_p = np.sqrt(np.sum(w * (pred_ranks - mu_p) ** 2))
    std_t = np.sqrt(np.sum(w * (true_ranks - mu_t) ** 2))

    if std_p <= 1e-12 or std_t <= 1e-12:
        return 0.0

    return max(cov / (std_p * std_t), 0.0)


def compute_cls(y_true, y_score, pe_eff, epsilon=0.01):
    """
    CLS = harmonic mean of PR-AUC and Weighted Spearman.
    """
    y_true = np.asarray(y_true, dtype=int)
    y_score = np.asarray(y_score, dtype=float)
    pe_eff = np.asarray(pe_eff, dtype=float)

    pr_auc = average_precision_score(y_true, y_score)

    weights = pe_eff + epsilon
    w_spear = weighted_spearman(y_score, pe_eff, weights)

    if pr_auc > 0 and w_spear > 0:
        cls = 2 * pr_auc * w_spear / (pr_auc + w_spear)
    else:
        cls = 0.0

    return cls, pr_auc, w_spear


def print_metric_summary(name, y_true, y_score, pe_eff):
    cls, pr_auc, w_spear = compute_cls(y_true, y_score, pe_eff)

    print(f"{name}")
    print("-" * 80)
    print(f"PR-AUC            : {pr_auc:.6f}")
    print(f"Weighted Spearman : {w_spear:.6f}")
    print(f"CLS               : {cls:.6f}")

    return {
        "model": name,
        "pr_auc": pr_auc,
        "weighted_spearman": w_spear,
        "cls": cls,
    }


# ------------------------------------------------------------
# 2. Rank normalization helpers
# ------------------------------------------------------------
def rank01(x):
    """
    Convert values to [0, 1] rank scale.
    Useful because CLS depends mostly on ranking.
    """
    s = pd.Series(x).astype(float)
    s = s.fillna(s.median())

    ranks = s.rank(method="average", pct=True).values

    if len(ranks) <= 1:
        return np.zeros_like(ranks, dtype=float)

    return (ranks - ranks.min()) / (ranks.max() - ranks.min() + 1e-12)


def safe_zscore(x):
    """
    Simple z-score with median NaN handling.
    """
    s = pd.Series(x).astype(float)
    s = s.fillna(s.median())
    arr = s.values

    std = np.std(arr)

    if std <= 1e-12:
        return np.zeros_like(arr, dtype=float)

    return (arr - np.mean(arr)) / std


def percentile_transform(train_scores, new_scores):
    """
    Map new_scores to percentile scale using train_scores distribution.

    This is useful for applying fold-trained scores to held-out family
    without ranking the held-out family internally.
    """
    train_scores = pd.Series(train_scores).astype(float)
    train_scores = train_scores.fillna(train_scores.median()).values

    new_scores = pd.Series(new_scores).astype(float)
    new_scores = new_scores.fillna(np.nanmedian(train_scores)).values

    sorted_train = np.sort(train_scores)
    ranks = np.searchsorted(sorted_train, new_scores, side="right")

    return ranks / max(len(sorted_train), 1)


# ------------------------------------------------------------
# 3. Family-level diagnostics
# ------------------------------------------------------------
def evaluate_by_family(score, model_name, df=train_df):
    """
    Compute family-level metrics for diagnostics.
    Families with only one class get NaN for PR-AUC / CLS.
    """
    rows = []

    for fam in df[FAMILY_COL].unique():
        idx = df[FAMILY_COL].values == fam

        y_fam = df.loc[idx, TARGET_COL].astype(int).values
        eff_fam = df.loc[idx, EFF_COL].astype(float).values
        score_fam = np.asarray(score, dtype=float)[idx]

        if len(np.unique(y_fam)) > 1:
            cls, pr_auc, w_spear = compute_cls(y_fam, score_fam, eff_fam)
        else:
            cls, pr_auc, w_spear = np.nan, np.nan, np.nan

        rows.append({
            "model": model_name,
            "family": fam,
            "n": int(idx.sum()),
            "n_active": int(y_fam.sum()),
            "pr_auc": pr_auc,
            "weighted_spearman": w_spear,
            "cls": cls,
            "mean_score": float(np.mean(score_fam)),
            "median_score": float(np.median(score_fam)),
        })

    return pd.DataFrame(rows).sort_values(["family", "model"])


def show_top_fp_fn(score, model_name, df=train_df, n=15):
    """
    Show top false positives and bottom false negatives.
    """
    tmp = df[[ID_COL, FAMILY_COL, TARGET_COL, EFF_COL]].copy()
    tmp["score"] = np.asarray(score, dtype=float)
    tmp["rank_score"] = rank01(tmp["score"].values)

    print("\n" + "=" * 80)
    print(f"Top false positives: {model_name}")
    print("=" * 80)
    display(
        tmp[tmp[TARGET_COL] == 0]
        .sort_values("score", ascending=False)
        .head(n)
    )

    print("\n" + "=" * 80)
    print(f"Bottom false negatives: {model_name}")
    print("=" * 80)
    display(
        tmp[tmp[TARGET_COL] == 1]
        .sort_values("score", ascending=True)
        .head(n)
    )


# ------------------------------------------------------------
# 4. LOFO splitter check
# ------------------------------------------------------------
logo = LeaveOneGroupOut()

print("=" * 80)
print("LOFO SPLIT CHECK")
print("=" * 80)

fold_rows = []

for fold, (train_idx, valid_idx) in enumerate(
    logo.split(train_df, y_active, groups=families),
    start=1
):
    holdout_family = families[valid_idx][0]

    fold_rows.append({
        "fold": fold,
        "holdout_family": holdout_family,
        "n_train": len(train_idx),
        "n_valid": len(valid_idx),
        "valid_active": int(y_active[valid_idx].sum()),
        "valid_inactive": int((y_active[valid_idx] == 0).sum()),
    })

fold_check = pd.DataFrame(fold_rows)
display(fold_check)

print("Ready for feature engineering.")
print("Metric/helper functions ready.")

LOFO SPLIT CHECK


,fold,holdout_family,n_train,n_valid,valid_active,valid_inactive
0,1,CRISPR-associated,52,5,0,5
1,2,Group_II_Intron,52,5,2,3
2,3,LTR_Retrotransposon,46,11,2,9
3,4,Other,52,5,0,5
4,5,Retron,45,12,5,7
5,6,Retroviral,39,18,12,6
6,7,Unclassified,56,1,0,1


Ready for feature engineering.
Metric/helper functions ready.


In [4]:
# ============================================================
# Step 3. Basic Feature Engineering Helpers
# Retroviral Wall Challenge Final Pipeline
# ============================================================

# ------------------------------------------------------------
# 1. Helper: safe feature access
# ------------------------------------------------------------
def has_cols(df, cols):
    return [c for c in cols if c in df.columns]


def safe_mean(df, cols, default=0.0):
    valid = has_cols(df, cols)
    if len(valid) == 0:
        return np.ones(len(df)) * default
    return df[valid].mean(axis=1).values


def safe_max(df, cols, default=0.0):
    valid = has_cols(df, cols)
    if len(valid) == 0:
        return np.ones(len(df)) * default
    return df[valid].max(axis=1).values


def safe_col(df, col, default=0.0):
    if col not in df.columns:
        return np.ones(len(df)) * default
    return df[col].astype(float).values


def add_missing_indicators(df, cols):
    df = df.copy()
    for col in cols:
        if col in df.columns:
            df[f"{col}_is_missing"] = df[col].isna().astype(float)
    return df


# ------------------------------------------------------------
# 2. Structure similarity features
# ------------------------------------------------------------
def add_similarity_features(df):
    """
    Add feature-based similarity summaries.
    No family-label memorization is used.
    """
    df = df.copy()

    canonical_cols = [
        "foldseek_TM_HIV1",
        "foldseek_TM_MMLV",
        "foldseek_TM_MMLVPE",
    ]

    noncanonical_cols = [
        "foldseek_TM_Retron",
        "foldseek_TM_Group2Intron",
        "foldseek_TM_Telomerase",
    ]

    canonical_cols = has_cols(df, canonical_cols)
    noncanonical_cols = has_cols(df, noncanonical_cols)

    if len(canonical_cols) > 0:
        df["canonical_similarity_max"] = df[canonical_cols].max(axis=1)
        df["canonical_similarity_mean"] = df[canonical_cols].mean(axis=1)

    if len(noncanonical_cols) > 0:
        df["noncanonical_similarity_max"] = df[noncanonical_cols].max(axis=1)
        df["noncanonical_similarity_mean"] = df[noncanonical_cols].mean(axis=1)

    if "canonical_similarity_max" in df.columns and "noncanonical_similarity_max" in df.columns:
        df["noncanonical_minus_canonical_max"] = (
            df["noncanonical_similarity_max"] - df["canonical_similarity_max"]
        )

    if "canonical_similarity_mean" in df.columns and "noncanonical_similarity_mean" in df.columns:
        df["noncanonical_minus_canonical_mean"] = (
            df["noncanonical_similarity_mean"] - df["canonical_similarity_mean"]
        )

    if "foldseek_TM_Retron" in df.columns and "canonical_similarity_max" in df.columns:
        df["retron_minus_canonical_max"] = (
            df["foldseek_TM_Retron"] - df["canonical_similarity_max"]
        )

    if "foldseek_TM_Retron" in df.columns and "canonical_similarity_mean" in df.columns:
        df["retron_minus_canonical_mean"] = (
            df["foldseek_TM_Retron"] - df["canonical_similarity_mean"]
        )

    if "foldseek_TM_Retron" in df.columns and "foldseek_TM_Group2Intron" in df.columns:
        df["retron_group2_similarity_mean"] = df[
            ["foldseek_TM_Retron", "foldseek_TM_Group2Intron"]
        ].mean(axis=1)

    if "protein_length_aa" in df.columns and "noncanonical_similarity_max" in df.columns:
        safe_len = df["protein_length_aa"].replace(0, np.nan)
        df["noncanonical_per_length"] = df["noncanonical_similarity_max"] / safe_len

    return df


# ------------------------------------------------------------
# 3. Prepare base engineered tables
# ------------------------------------------------------------
train_base = add_similarity_features(train_df)

if test_df is not None:
    test_base = add_similarity_features(test_df)
else:
    test_base = None

print("=" * 80)
print("BASIC FEATURE ENGINEERING SUMMARY")
print("=" * 80)
print("train_base shape:", train_base.shape)

if test_base is not None:
    print("test_base shape:", test_base.shape)

added_cols = [
    "canonical_similarity_max",
    "canonical_similarity_mean",
    "noncanonical_similarity_max",
    "noncanonical_similarity_mean",
    "noncanonical_minus_canonical_max",
    "noncanonical_minus_canonical_mean",
    "retron_minus_canonical_max",
    "retron_minus_canonical_mean",
    "retron_group2_similarity_mean",
    "noncanonical_per_length",
]

print("\nAdded similarity columns:")
print([c for c in added_cols if c in train_base.columns])

display(train_base[[ID_COL, FAMILY_COL, TARGET_COL, EFF_COL]].head())

print("\nReady for block-score modeling.")

BASIC FEATURE ENGINEERING SUMMARY
train_base shape: (57, 81)
test_base shape: (57, 79)

Added similarity columns:
['canonical_similarity_max', 'canonical_similarity_mean', 'noncanonical_similarity_max', 'noncanonical_similarity_mean', 'noncanonical_minus_canonical_max', 'noncanonical_minus_canonical_mean', 'retron_minus_canonical_max', 'retron_minus_canonical_mean', 'retron_group2_similarity_mean', 'noncanonical_per_length']


,rt_name,rt_family,active,pe_efficiency_pct
0,A.platensis-Cas1-RT,CRISPR-associated,0,0.0
1,CRISPRRT2-RT,CRISPR-associated,0,0.0
2,FuRT-Cas1-RT,CRISPR-associated,0,0.0
3,Med-CasRT,CRISPR-associated,0,0.0
4,ROSE-CRISPR-RT,CRISPR-associated,0,0.0



Ready for block-score modeling.


In [5]:
# ============================================================
# Step 4-1. Original Block Score Generation
# Reproduces block_score_diagnostic_table in memory
# ============================================================

# ------------------------------------------------------------
# 0. Compatibility aliases
# ------------------------------------------------------------
if "COMP_DIR" not in globals():
    if "DATA_DIR" in globals():
        COMP_DIR = DATA_DIR
    else:
        COMP_DIR = "/kaggle/input/competitions/retroviral-challenge-predict"

DATA_DIR = COMP_DIR

# IMPORTANT:
# Use raw training dataframe.
# Do NOT use train_fe here, because train_fe may already contain
# gate/correction features and can contaminate the original block-score models.
if "train_df" in globals():
    train = train_df.copy()
else:
    train = pd.read_csv(f"{DATA_DIR}/train.csv")

id_col = ID_COL if "ID_COL" in globals() else "rt_name"
target_col = TARGET_COL if "TARGET_COL" in globals() else "active"
eff_col = EFF_COL if "EFF_COL" in globals() else "pe_efficiency_pct"
group_col = FAMILY_COL if "FAMILY_COL" in globals() else "rt_family"

y = train[target_col].astype(int).copy()
eff = train[eff_col].astype(float).copy()
groups = train[group_col].copy()

print("=" * 80)
print("BLOCK SCORE INPUT")
print("=" * 80)
print("train shape:", train.shape)
print("DATA_DIR:", DATA_DIR)
display(train[[id_col, group_col, target_col, eff_col]].head())


# ------------------------------------------------------------
# 1. Metrics
# ------------------------------------------------------------
def weighted_spearman(pred, true_eff, weights):
    pred = np.asarray(pred, dtype=float)
    true_eff = np.asarray(true_eff, dtype=float)
    weights = np.asarray(weights, dtype=float)

    pred_ranks = np.argsort(np.argsort(pred)).astype(float)
    true_ranks = np.argsort(np.argsort(true_eff)).astype(float)

    w = weights / weights.sum()

    mu_p = np.dot(w, pred_ranks)
    mu_t = np.dot(w, true_ranks)

    cov = np.sum(w * (pred_ranks - mu_p) * (true_ranks - mu_t))
    std_p = np.sqrt(np.sum(w * (pred_ranks - mu_p) ** 2))
    std_t = np.sqrt(np.sum(w * (true_ranks - mu_t) ** 2))

    if std_p <= 1e-12 or std_t <= 1e-12:
        return 0.0

    return max(cov / (std_p * std_t), 0.0)


def compute_cls(y_true, y_score, pe_eff):
    pr_auc = average_precision_score(y_true, y_score)
    w_spear = weighted_spearman(y_score, pe_eff, pe_eff + 0.01)

    if pr_auc > 0 and w_spear > 0:
        cls = 2 * pr_auc * w_spear / (pr_auc + w_spear)
    else:
        cls = 0.0

    return cls, pr_auc, w_spear


def evaluate_score(name, score):
    cls, pr_auc, w_spear = compute_cls(y, score, eff)
    return {
        "model": name,
        "pr_auc": pr_auc,
        "weighted_spearman": w_spear,
        "cls": cls,
    }


def evaluate_by_family(score, model_name):
    rows = []

    for fam in train[group_col].unique():
        idx = train[group_col] == fam

        y_fam = y[idx]
        eff_fam = eff[idx]
        score_fam = np.asarray(score)[idx]

        if y_fam.nunique() > 1:
            cls, pr_auc, w_spear = compute_cls(y_fam, score_fam, eff_fam)
        else:
            cls, pr_auc, w_spear = np.nan, np.nan, np.nan

        rows.append({
            "model": model_name,
            "family": fam,
            "n": int(idx.sum()),
            "n_active": int(y_fam.sum()),
            "pr_auc": pr_auc,
            "weighted_spearman": w_spear,
            "cls": cls,
            "mean_score": float(np.mean(score_fam)),
            "median_score": float(np.median(score_fam)),
        })

    return pd.DataFrame(rows)


def rank01(x):
    x = np.asarray(x, dtype=float)
    ranks = np.argsort(np.argsort(x)).astype(float)
    if len(ranks) <= 1:
        return ranks
    return ranks / (len(ranks) - 1)


# ------------------------------------------------------------
# 2. Add simple engineered similarity features
# ------------------------------------------------------------
def add_simple_rt_similarity_features(df):
    df = df.copy()

    canonical_cols = [
        "foldseek_TM_HIV1",
        "foldseek_TM_MMLV",
        "foldseek_TM_MMLVPE",
    ]

    noncanonical_cols = [
        "foldseek_TM_Retron",
        "foldseek_TM_Group2Intron",
        "foldseek_TM_Telomerase",
    ]

    canonical_cols = [c for c in canonical_cols if c in df.columns]
    noncanonical_cols = [c for c in noncanonical_cols if c in df.columns]

    if len(canonical_cols) > 0:
        df["canonical_similarity_max"] = df[canonical_cols].max(axis=1)
        df["canonical_similarity_mean"] = df[canonical_cols].mean(axis=1)

    if len(noncanonical_cols) > 0:
        df["noncanonical_similarity_max"] = df[noncanonical_cols].max(axis=1)
        df["noncanonical_similarity_mean"] = df[noncanonical_cols].mean(axis=1)

    if "canonical_similarity_max" in df.columns and "noncanonical_similarity_max" in df.columns:
        df["noncanonical_minus_canonical_max"] = (
            df["noncanonical_similarity_max"] - df["canonical_similarity_max"]
        )

    if "canonical_similarity_mean" in df.columns and "noncanonical_similarity_mean" in df.columns:
        df["noncanonical_minus_canonical_mean"] = (
            df["noncanonical_similarity_mean"] - df["canonical_similarity_mean"]
        )

    if "foldseek_TM_Retron" in df.columns and "canonical_similarity_max" in df.columns:
        df["retron_minus_canonical_max"] = (
            df["foldseek_TM_Retron"] - df["canonical_similarity_max"]
        )

    if "foldseek_TM_Retron" in df.columns and "canonical_similarity_mean" in df.columns:
        df["retron_minus_canonical_mean"] = (
            df["foldseek_TM_Retron"] - df["canonical_similarity_mean"]
        )

    if "foldseek_TM_Retron" in df.columns and "foldseek_TM_Group2Intron" in df.columns:
        df["retron_group2_similarity_mean"] = df[
            ["foldseek_TM_Retron", "foldseek_TM_Group2Intron"]
        ].mean(axis=1)

    if "protein_length_aa" in df.columns and "noncanonical_similarity_max" in df.columns:
        safe_len = df["protein_length_aa"].replace(0, np.nan)
        df["noncanonical_per_length"] = df["noncanonical_similarity_max"] / safe_len

    return df


train_engineered = add_simple_rt_similarity_features(train)


# ------------------------------------------------------------
# 3. Define original feature blocks
# ------------------------------------------------------------
feature_blocks = {
    "global_structure": [
        "foldseek_best_TM",
        "foldseek_best_LDDT",
        "foldseek_best_fident",
    ],

    "canonical_retroviral_similarity": [
        "foldseek_TM_HIV1",
        "foldseek_TM_MMLV",
        "foldseek_TM_MMLVPE",
        "canonical_similarity_max",
        "canonical_similarity_mean",
    ],

    "noncanonical_similarity": [
        "foldseek_TM_Retron",
        "foldseek_TM_Group2Intron",
        "foldseek_TM_Telomerase",
        "noncanonical_similarity_max",
        "noncanonical_similarity_mean",
        "noncanonical_minus_canonical_max",
        "noncanonical_minus_canonical_mean",
        "retron_minus_canonical_max",
        "retron_minus_canonical_mean",
        "retron_group2_similarity_mean",
        "noncanonical_per_length",
    ],

    "active_site_geometry": [
        "triad_found_bin",
        "D1_D2_dist",
        "D2_D3_dist",
        "triad_best_rmsd",
        "hairpin_pass",
        "hairpin_confidence",
        "yxdd_y_is_strand",
        "yxdd_x_is_turn",
        "yxdd_d1_is_turn",
        "yxdd_d2_is_strand",
        "yxdd_hydrophobic_fraction",
        "yxdd_mean_hydrophobicity",
        "yxdd_std_hydrophobicity",
    ],

    "thermal_stability": [
        "t40_raw",
        "t45_raw",
        "t50_raw",
        "t55_raw",
        "t60_raw",
        "t65_raw",
        "t70_raw",
        "t75_raw",
        "t80_raw",
        "thermophilicity_num",
    ],

    "physicochemical": [
        "perplexity",
        "log_likelihood",
        "instability_index",
        "gravy",
        "camsol",
        "net_charge",
        "helix_beta_ratio",
        "protein_length_aa",
    ],

    "contacts_and_pocket": [
        "hydrophobic_per_res",
        "salt_per_res",
        "hbonds_per_res",
        "pocket_hydrophobic_per_res",
        "pocket_hbonds_per_res",
    ],

    "exposure_quality": [
        "sasa_per_res",
        "apolar_ratio",
        "mean_rsasa",
        "pocket_mean_rsasa",
        "nterm_avg_sasa",
        "rama_fav",
        "rama_out",
        "n_residues",
    ],

    "electrostatics": [
        "overall_mean_pot",
        "fingers_mean_pot",
        "motif12_mean_pot",
        "lpqg_sp_3A_mean_pot",
        "yxdd_5A_mean_pot",
        "palm_mean_pot",
        "thumb_mean_pot",
        "connection_mean_pot",
    ],
}

clean_blocks = {}

for block_name, cols in feature_blocks.items():
    valid_cols = [
        c for c in cols
        if c in train_engineered.columns
        and pd.api.types.is_numeric_dtype(train_engineered[c])
    ]
    clean_blocks[block_name] = valid_cols

print("=" * 80)
print("FEATURE BLOCKS")
print("=" * 80)

for block_name, cols in clean_blocks.items():
    print(f"{block_name:35s}: {len(cols):2d} features")
    print(cols)
    print()


# ------------------------------------------------------------
# 4. OOF helper for numeric block model
# ------------------------------------------------------------
def get_block_oof(df, feature_cols, model_name, c_value=0.1):
    X = df[feature_cols].copy()

    model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            max_iter=5000,
            class_weight="balanced",
            penalty="l2",
            C=c_value,
            random_state=42
        ))
    ])

    logo = LeaveOneGroupOut()
    oof = np.zeros(len(df))
    fold_rows = []

    for fold, (tr_idx, va_idx) in enumerate(
        logo.split(X, y, groups=groups),
        start=1
    ):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]
        eff_va = eff.iloc[va_idx]
        fam = groups.iloc[va_idx].iloc[0]

        model.fit(X_tr, y_tr)
        pred = model.predict_proba(X_va)[:, 1]
        oof[va_idx] = pred

        if y_va.nunique() > 1:
            cls, pr_auc, w_spear = compute_cls(y_va, pred, eff_va)
        else:
            cls, pr_auc, w_spear = np.nan, np.nan, np.nan

        fold_rows.append({
            "model": model_name,
            "fold": fold,
            "holdout_family": fam,
            "n_valid": len(va_idx),
            "n_active_valid": int(y_va.sum()),
            "pr_auc": pr_auc,
            "weighted_spearman": w_spear,
            "cls": cls,
            "mean_score": float(np.mean(pred)),
            "median_score": float(np.median(pred)),
        })

    return oof, pd.DataFrame(fold_rows)


# ------------------------------------------------------------
# 5. OOF helper for ESM2 PCA model
# ------------------------------------------------------------
def get_embedding_pca_oof(train_df, data_dir, n_components=3, c_value=0.1):
    emb_path = os.path.join(data_dir, "esm2_embeddings.npz")

    if not os.path.exists(emb_path):
        raise FileNotFoundError(f"Missing embedding file: {emb_path}")

    emb_data = np.load(emb_path, allow_pickle=True)
    emb_dict = dict(zip(emb_data["names"], emb_data["embeddings"]))

    X_emb_full = np.vstack([emb_dict[name] for name in train_df[id_col]])

    X = pd.DataFrame(
        X_emb_full,
        columns=[f"esm2_{i}" for i in range(X_emb_full.shape[1])]
    )

    model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("pca", PCA(n_components=n_components, random_state=42)),
        ("clf", LogisticRegression(
            max_iter=5000,
            class_weight="balanced",
            penalty="l2",
            C=c_value,
            random_state=42
        ))
    ])

    logo = LeaveOneGroupOut()
    oof = np.zeros(len(train_df))
    fold_rows = []

    for fold, (tr_idx, va_idx) in enumerate(
        logo.split(X, y, groups=groups),
        start=1
    ):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]
        eff_va = eff.iloc[va_idx]
        fam = groups.iloc[va_idx].iloc[0]

        model.fit(X_tr, y_tr)
        pred = model.predict_proba(X_va)[:, 1]
        oof[va_idx] = pred

        if y_va.nunique() > 1:
            cls, pr_auc, w_spear = compute_cls(y_va, pred, eff_va)
        else:
            cls, pr_auc, w_spear = np.nan, np.nan, np.nan

        fold_rows.append({
            "model": f"embedding_pca{n_components}",
            "fold": fold,
            "holdout_family": fam,
            "n_valid": len(va_idx),
            "n_active_valid": int(y_va.sum()),
            "pr_auc": pr_auc,
            "weighted_spearman": w_spear,
            "cls": cls,
            "mean_score": float(np.mean(pred)),
            "median_score": float(np.median(pred)),
        })

    return oof, pd.DataFrame(fold_rows)


# ------------------------------------------------------------
# 6. Build OOF score for each block
# ------------------------------------------------------------
block_scores = {}
block_fold_tables = []
block_summary_rows = []

for block_name, cols in clean_blocks.items():
    if len(cols) == 0:
        continue

    print(f"Training block model: {block_name} ({len(cols)} features)")

    oof, fold_df = get_block_oof(
        train_engineered,
        cols,
        model_name=block_name,
        c_value=0.1
    )

    block_scores[block_name] = oof
    block_fold_tables.append(fold_df)
    block_summary_rows.append(evaluate_score(block_name, oof))

# ESM2 PCA block
print("Training embedding PCA3 block model")

embedding_oof, embedding_fold_df = get_embedding_pca_oof(
    train,
    DATA_DIR,
    n_components=3,
    c_value=0.1
)

block_scores["embedding_pca3"] = embedding_oof
block_fold_tables.append(embedding_fold_df)
block_summary_rows.append(evaluate_score("embedding_pca3", embedding_oof))

block_summary_df = pd.DataFrame(block_summary_rows).sort_values("cls", ascending=False)
block_fold_df = pd.concat(block_fold_tables, ignore_index=True)

print("\n" + "=" * 80)
print("BLOCK MODEL SUMMARY")
print("=" * 80)
display(block_summary_df)

print("\n" + "=" * 80)
print("BLOCK FOLD RESULTS")
print("=" * 80)
display(block_fold_df.sort_values(["holdout_family", "model"]))


# ------------------------------------------------------------
# 7. Family-level comparison for each block
# ------------------------------------------------------------
family_tables = []

for block_name, score in block_scores.items():
    family_tables.append(evaluate_by_family(score, block_name))

block_family_df = pd.concat(family_tables, ignore_index=True)

print("\n" + "=" * 80)
print("BLOCK FAMILY-LEVEL PERFORMANCE")
print("=" * 80)
display(
    block_family_df.sort_values(["family", "cls"], ascending=[True, False])
)


# ------------------------------------------------------------
# 8. Blend block scores
# ------------------------------------------------------------
score_names = list(block_scores.keys())
score_matrix = np.column_stack([rank01(block_scores[name]) for name in score_names])

print("\nScore blocks used for blending:")
for i, name in enumerate(score_names):
    print(i, name)

rng = np.random.default_rng(42)
blend_rows = []

# Individual block scores
for i, name in enumerate(score_names):
    w = np.zeros(len(score_names))
    w[i] = 1.0
    score = score_matrix @ w
    cls, pr_auc, w_spear = compute_cls(y, score, eff)

    row = {
        "blend_type": "single_block",
        "pr_auc": pr_auc,
        "weighted_spearman": w_spear,
        "cls": cls,
    }
    for j, score_name in enumerate(score_names):
        row[f"w_{score_name}"] = w[j]
    blend_rows.append(row)

# Average handcrafted blocks + embedding sweep
handcrafted_block_names = [n for n in score_names if n != "embedding_pca3"]

handcrafted_avg = np.mean(
    np.column_stack([rank01(block_scores[n]) for n in handcrafted_block_names]),
    axis=1
)

embedding_rank = rank01(block_scores["embedding_pca3"])

for w_emb in np.arange(0.0, 1.01, 0.01):
    score = (1 - w_emb) * handcrafted_avg + w_emb * embedding_rank
    cls, pr_auc, w_spear = compute_cls(y, score, eff)

    row = {
        "blend_type": "avg_handcrafted_blocks_plus_embedding",
        "pr_auc": pr_auc,
        "weighted_spearman": w_spear,
        "cls": cls,
    }
    for score_name in score_names:
        row[f"w_{score_name}"] = np.nan
    row["w_avg_handcrafted_blocks"] = 1 - w_emb
    row["w_embedding_pca3_manual"] = w_emb
    blend_rows.append(row)

# Random sparse-ish blends
for k in range(5000):
    weights = rng.dirichlet(alpha=np.ones(len(score_names)) * 0.35)
    score = score_matrix @ weights
    cls, pr_auc, w_spear = compute_cls(y, score, eff)

    row = {
        "blend_type": "dirichlet_random",
        "pr_auc": pr_auc,
        "weighted_spearman": w_spear,
        "cls": cls,
    }

    for j, score_name in enumerate(score_names):
        row[f"w_{score_name}"] = weights[j]

    blend_rows.append(row)

blend_df = pd.DataFrame(blend_rows).sort_values("cls", ascending=False).reset_index(drop=True)

print("\n" + "=" * 80)
print("TOP BLOCK-SCORE BLENDS")
print("=" * 80)
display(blend_df.head(30))

best_blend = blend_df.iloc[0]
best_weights = np.array([best_blend[f"w_{name}"] for name in score_names], dtype=float)

if np.any(pd.isna(best_weights)):
    w_emb = float(best_blend["w_embedding_pca3_manual"])
    best_block_blend_score = (1 - w_emb) * handcrafted_avg + w_emb * embedding_rank
else:
    best_block_blend_score = score_matrix @ best_weights


# ------------------------------------------------------------
# 9. Compare against full engineered + embedding baseline
# ------------------------------------------------------------
drop_cols = [
    id_col,
    "sequence",
    target_col,
    eff_col,
    group_col,
    "yxdd_seq",
    "connection_mean_pot",
]

engineered_cols = [
    c for c in train_engineered.columns
    if c not in drop_cols
    and pd.api.types.is_numeric_dtype(train_engineered[c])
]

full_engineered_oof, full_engineered_fold_df = get_block_oof(
    train_engineered,
    engineered_cols,
    model_name="full_engineered_handcrafted",
    c_value=0.1
)

full_two_rows = []

for w_emb in np.arange(0.00, 1.01, 0.01):
    score = (1 - w_emb) * full_engineered_oof + w_emb * embedding_oof
    cls, pr_auc, w_spear = compute_cls(y, score, eff)

    full_two_rows.append({
        "embedding_weight": round(float(w_emb), 2),
        "engineered_weight": round(float(1 - w_emb), 2),
        "pr_auc": pr_auc,
        "weighted_spearman": w_spear,
        "cls": cls,
    })

full_two_df = pd.DataFrame(full_two_rows).sort_values("cls", ascending=False).reset_index(drop=True)

best_w_emb = float(full_two_df.loc[0, "embedding_weight"])
best_full_two_score = (1 - best_w_emb) * full_engineered_oof + best_w_emb * embedding_oof

comparison_rows = []
comparison_rows.append(evaluate_score("full_engineered_only", full_engineered_oof))
comparison_rows.append(evaluate_score("embedding_pca3_only", embedding_oof))
comparison_rows.append(
    evaluate_score(
        f"full_two_branch_eng{1-best_w_emb:.2f}_emb{best_w_emb:.2f}",
        best_full_two_score
    )
)
comparison_rows.append(evaluate_score("best_block_score_blend", best_block_blend_score))

comparison_df = pd.DataFrame(comparison_rows).sort_values("cls", ascending=False)

print("\n" + "=" * 80)
print("FINAL COMPARISON: FULL VS BLOCK-SCORE BLEND")
print("=" * 80)
display(comparison_df)

print("\n" + "=" * 80)
print("FULL TWO-BRANCH BLEND SEARCH")
print("=" * 80)
display(full_two_df.head(15))


# ------------------------------------------------------------
# 10. Family-level diagnostics for best models
# ------------------------------------------------------------
final_family_compare = pd.concat([
    evaluate_by_family(best_full_two_score, "full_two_branch_best"),
    evaluate_by_family(best_block_blend_score, "best_block_score_blend"),
    evaluate_by_family(embedding_oof, "embedding_pca3"),
], ignore_index=True)

print("\n" + "=" * 80)
print("FINAL FAMILY-LEVEL COMPARISON")
print("=" * 80)
display(
    final_family_compare.sort_values(["family", "model"])
)


# ------------------------------------------------------------
# 11. Diagnostic table
# ------------------------------------------------------------
diagnostic_df = train[[id_col, group_col, target_col, eff_col]].copy()

for name, score in block_scores.items():
    diagnostic_df[f"score_{name}"] = score

diagnostic_df["score_full_engineered"] = full_engineered_oof
diagnostic_df["score_full_two_branch_best"] = best_full_two_score
diagnostic_df["score_best_block_blend"] = best_block_blend_score

print("\n" + "=" * 80)
print("RETRON DIAGNOSTICS")
print("=" * 80)
display(
    diagnostic_df[diagnostic_df[group_col] == "Retron"]
    .sort_values("score_best_block_blend", ascending=False)
)

print("\n" + "=" * 80)
print("TOP 20 BY BEST BLOCK BLEND")
print("=" * 80)
display(
    diagnostic_df.sort_values("score_best_block_blend", ascending=False).head(20)
)

print("\nStep 4-1 outputs ready.")
print("diagnostic_df:", diagnostic_df.shape)

block_score_cols = [c for c in diagnostic_df.columns if c.startswith("score_")]
print("Block score columns:")
print(block_score_cols)

BLOCK SCORE INPUT
train shape: (57, 71)
DATA_DIR: /kaggle/input/competitions/retroviral-challenge-predict


,rt_name,rt_family,active,pe_efficiency_pct
0,A.platensis-Cas1-RT,CRISPR-associated,0,0.0
1,CRISPRRT2-RT,CRISPR-associated,0,0.0
2,FuRT-Cas1-RT,CRISPR-associated,0,0.0
3,Med-CasRT,CRISPR-associated,0,0.0
4,ROSE-CRISPR-RT,CRISPR-associated,0,0.0


FEATURE BLOCKS
global_structure                   :  3 features
['foldseek_best_TM', 'foldseek_best_LDDT', 'foldseek_best_fident']

canonical_retroviral_similarity    :  5 features
['foldseek_TM_HIV1', 'foldseek_TM_MMLV', 'foldseek_TM_MMLVPE', 'canonical_similarity_max', 'canonical_similarity_mean']

noncanonical_similarity            : 11 features
['foldseek_TM_Retron', 'foldseek_TM_Group2Intron', 'foldseek_TM_Telomerase', 'noncanonical_similarity_max', 'noncanonical_similarity_mean', 'noncanonical_minus_canonical_max', 'noncanonical_minus_canonical_mean', 'retron_minus_canonical_max', 'retron_minus_canonical_mean', 'retron_group2_similarity_mean', 'noncanonical_per_length']

active_site_geometry               : 13 features
['triad_found_bin', 'D1_D2_dist', 'D2_D3_dist', 'triad_best_rmsd', 'hairpin_pass', 'hairpin_confidence', 'yxdd_y_is_strand', 'yxdd_x_is_turn', 'yxdd_d1_is_turn', 'yxdd_d2_is_strand', 'yxdd_hydrophobic_fraction', 'yxdd_mean_hydrophobicity', 'yxdd_std_hydrophobicity'

,model,pr_auc,weighted_spearman,cls
1,canonical_retroviral_similarity,0.588561,0.616699,0.602301
9,embedding_pca3,0.525039,0.634141,0.574455
3,active_site_geometry,0.396219,0.661486,0.495588
2,noncanonical_similarity,0.435943,0.562293,0.491122
6,contacts_and_pocket,0.367954,0.438089,0.399970
5,physicochemical,0.329390,0.398269,0.360570
7,exposure_quality,0.303041,0.426957,0.354482
8,electrostatics,0.377076,0.305500,0.337535
0,global_structure,0.663802,0.169853,0.270492
4,thermal_stability,0.302208,0.232178,0.262604



BLOCK FOLD RESULTS


...


RETRON DIAGNOSTICS


,rt_name,rt_family,active,pe_efficiency_pct,score_global_structure,score_canonical_retroviral_similarity,score_noncanonical_similarity,score_active_site_geometry,score_thermal_stability,score_physicochemical,score_contacts_and_pocket,score_exposure_quality,score_electrostatics,score_embedding_pca3,score_full_engineered,score_full_two_branch_best,score_best_block_blend
31,Rs-RT,Retron,1,2.5,0.480395,0.164155,0.200437,0.353303,0.291813,0.290940,0.555674,0.371880,0.180922,0.144313,0.068742,0.108039,0.522384
36,Vp96-RT,Retron,1,8.0,0.735205,0.266355,0.294052,0.297929,0.316831,0.424738,0.413680,0.308064,0.159789,0.199359,0.222229,0.210337,0.461278
32,SEN1-RT,Retron,0,0.0,0.647530,0.266852,0.288107,0.331283,0.340790,0.411361,0.374886,0.329566,0.266283,0.228313,0.262686,0.244812,0.432961
37,"retronE,coli-RT",Retron,0,0.0,0.777342,0.257641,0.280502,0.469578,0.386559,0.467958,0.320193,0.294500,0.562677,0.135763,0.404667,0.264837,0.373297
30,Retron86-RT,Retron,0,0.0,0.778110,0.232965,0.255102,0.541375,0.369972,0.464673,0.312516,0.315568,0.515804,0.135255,0.480049,0.300756,0.366073
35,Vc95-RT,Retron,1,1.5,0.679729,0.257065,0.288313,0.140364,0.331942,0.332918,0.302313,0.308252,0.354065,0.198101,0.234812,0.215723,0.336192
28,Mx65-RT,Retron,0,0.0,0.418302,0.209023,0.315520,0.257394,0.285741,0.347199,0.191271,0.140281,0.344604,0.202050,0.054353,0.131156,0.204847
33,Sau1-RT,Retron,0,0.0,0.341028,0.264017,0.362808,0.164820,0.290172,0.369263,0.187509,0.206714,0.355343,0.270562,0.066294,0.172514,0.201721
29,Ne144-RT,Retron,1,0.5,0.357575,0.220529,0.313044,0.219487,0.291486,0.342474,0.163325,0.132758,0.444541,0.249827,0.052118,0.154927,0.172531
27,Ec67-RT,Retron,0,0.0,0.266994,0.185920,0.307734,0.412218,0.307682,0.288979,0.298029,0.525433,0.372738,0.175137,0.094674,0.136515,0.135724



TOP 20 BY BEST BLOCK BLEND


,rt_name,rt_family,active,pe_efficiency_pct,score_global_structure,score_canonical_retroviral_similarity,score_noncanonical_similarity,score_active_site_geometry,score_thermal_stability,score_physicochemical,score_contacts_and_pocket,score_exposure_quality,score_electrostatics,score_embedding_pca3,score_full_engineered,score_full_two_branch_best,score_best_block_blend
47,MMLV-RT,Retroviral,1,41.0,0.781509,0.616030,0.450872,0.593878,0.445792,0.491298,0.479273,0.416475,0.445555,0.507091,0.819783,0.657183,0.786189
55,WMSV-RT,Retroviral,1,23.5,0.739003,0.615166,0.525261,0.525365,0.296424,0.471151,0.465385,0.424770,0.641871,0.541492,0.597174,0.568219,0.760768
39,AVIRE-RT,Retroviral,1,23.0,0.701341,0.612504,0.507740,0.472307,0.384256,0.416359,0.468789,0.428841,0.310364,0.547038,0.730335,0.635021,0.758563
17,Tf1-RT,LTR_Retrotransposon,1,34.0,0.408009,0.661611,0.640577,0.575124,0.597591,0.398647,0.566031,0.400488,0.651633,0.562897,0.522913,0.543704,0.749475
46,KORV-RT,Retroviral,1,26.5,0.741137,0.613475,0.519808,0.518171,0.323119,0.468882,0.459338,0.431924,0.439594,0.538834,0.669570,0.601588,0.745608
50,PERV-RT,Retroviral,1,21.0,0.741771,0.613909,0.523598,0.564821,0.342815,0.466910,0.457977,0.423085,0.526455,0.536393,0.677474,0.604112,0.740317
16,Ngaro-RT,LTR_Retrotransposon,0,0.0,0.301009,0.547448,0.535646,0.398582,0.646641,0.541942,0.675482,0.771039,0.481185,0.769010,0.396079,0.590003,0.740028
43,GALV-RT,Retroviral,1,5.5,0.737848,0.614543,0.516874,0.514701,0.309732,0.449850,0.463565,0.426314,0.664501,0.534850,0.611959,0.571862,0.732925
19,Ty3-RT,LTR_Retrotransposon,1,9.0,0.394330,0.654112,0.618679,0.346531,0.608700,0.425727,0.577912,0.487737,0.557203,0.520085,0.361435,0.443933,0.722314
22,CaMV-RT,Other,0,0.0,0.419403,0.625899,0.580344,0.320382,0.556922,0.460317,0.572335,0.430686,0.504731,0.502286,0.394469,0.450534,0.715135



Step 4-1 outputs ready.
diagnostic_df: (57, 17)
Block score columns:
['score_global_structure', 'score_canonical_retroviral_similarity', 'score_noncanonical_similarity', 'score_active_site_geometry', 'score_thermal_stability', 'score_physicochemical', 'score_contacts_and_pocket', 'score_exposure_quality', 'score_electrostatics', 'score_embedding_pca3', 'score_full_engineered', 'score_full_two_branch_best', 'score_best_block_blend']


In [6]:
# ============================================================
# Step 4-2. Original Structure-Aware Soft Scoring
# In-memory version for final submission notebook
# Uses diagnostic_df from Step 4-1 directly
# ============================================================

# ------------------------------------------------------------
# 0. Compatibility setup
# ------------------------------------------------------------
if "COMP_DIR" not in globals():
    if "DATA_DIR" in globals():
        COMP_DIR = DATA_DIR
    else:
        COMP_DIR = "/kaggle/input/competitions/retroviral-challenge-predict"

DATA_DIR = COMP_DIR

# Use existing train dataframe from Step 1.
# Do NOT reload or depend on intermediate CSV files.
if "train_df" in globals():
    train = train_df.copy()
else:
    train = pd.read_csv(f"{DATA_DIR}/train.csv")

id_col = ID_COL if "ID_COL" in globals() else "rt_name"
target_col = TARGET_COL if "TARGET_COL" in globals() else "active"
eff_col = EFF_COL if "EFF_COL" in globals() else "pe_efficiency_pct"
group_col = FAMILY_COL if "FAMILY_COL" in globals() else "rt_family"

y = train[target_col].astype(int).values
eff = train[eff_col].astype(float).values
groups = train[group_col].values

print("=" * 80)
print("STRUCTURE-AWARE INPUT")
print("=" * 80)
print("train:", train.shape)
display(train[group_col].value_counts())

# Step 4-1 must have created diagnostic_df.
if "diagnostic_df" not in globals():
    raise ValueError(
        "diagnostic_df not found. Run Step 4-1 block score generation first."
    )

block_diag = diagnostic_df.copy()

print("\nBlock diagnostic input:", block_diag.shape)
print("Block score columns:")
print([c for c in block_diag.columns if c.startswith("score_")])


# ============================================================
# 1. Small helper
# ============================================================
def sigmoid(x):
    return 1 / (1 + np.exp(-x))


# ============================================================
# 2. Add engineered similarity features
# ============================================================
def add_simple_rt_similarity_features(df):
    df = df.copy()

    canonical_cols = [
        "foldseek_TM_HIV1",
        "foldseek_TM_MMLV",
        "foldseek_TM_MMLVPE",
    ]

    noncanonical_cols = [
        "foldseek_TM_Retron",
        "foldseek_TM_Group2Intron",
        "foldseek_TM_Telomerase",
    ]

    canonical_cols = [c for c in canonical_cols if c in df.columns]
    noncanonical_cols = [c for c in noncanonical_cols if c in df.columns]

    if len(canonical_cols) > 0:
        df["canonical_similarity_max"] = df[canonical_cols].max(axis=1)
        df["canonical_similarity_mean"] = df[canonical_cols].mean(axis=1)

    if len(noncanonical_cols) > 0:
        df["noncanonical_similarity_max"] = df[noncanonical_cols].max(axis=1)
        df["noncanonical_similarity_mean"] = df[noncanonical_cols].mean(axis=1)

    if "canonical_similarity_max" in df.columns and "noncanonical_similarity_max" in df.columns:
        df["noncanonical_minus_canonical_max"] = (
            df["noncanonical_similarity_max"] - df["canonical_similarity_max"]
        )

    if "canonical_similarity_mean" in df.columns and "noncanonical_similarity_mean" in df.columns:
        df["noncanonical_minus_canonical_mean"] = (
            df["noncanonical_similarity_mean"] - df["canonical_similarity_mean"]
        )

    if "foldseek_TM_Retron" in df.columns and "canonical_similarity_max" in df.columns:
        df["retron_minus_canonical_max"] = (
            df["foldseek_TM_Retron"] - df["canonical_similarity_max"]
        )

    if "foldseek_TM_Retron" in df.columns and "canonical_similarity_mean" in df.columns:
        df["retron_minus_canonical_mean"] = (
            df["foldseek_TM_Retron"] - df["canonical_similarity_mean"]
        )

    if "foldseek_TM_Retron" in df.columns and "foldseek_TM_Group2Intron" in df.columns:
        df["retron_group2_similarity_mean"] = df[
            ["foldseek_TM_Retron", "foldseek_TM_Group2Intron"]
        ].mean(axis=1)

    if "protein_length_aa" in df.columns and "noncanonical_similarity_max" in df.columns:
        safe_len = df["protein_length_aa"].replace(0, np.nan)
        df["noncanonical_per_length"] = df["noncanonical_similarity_max"] / safe_len

    return df


train_engineered = add_simple_rt_similarity_features(train)


# ============================================================
# 3. Selected feature sets
# ============================================================
selected_handcrafted_v1 = [
    # FoldSeek / structural similarity
    "foldseek_best_TM",
    "foldseek_best_LDDT",
    "foldseek_best_fident",
    "foldseek_TM_HIV1",
    "foldseek_TM_MMLV",
    "foldseek_TM_MMLVPE",
    "foldseek_TM_Retron",
    "foldseek_TM_Group2Intron",
    "foldseek_TM_Telomerase",

    # Catalytic / active-site geometry
    "triad_found_bin",
    "D1_D2_dist",
    "D2_D3_dist",
    "triad_best_rmsd",
    "hairpin_pass",
    "hairpin_confidence",

    # Physicochemical
    "perplexity",
    "instability_index",
    "gravy",
    "camsol",
    "net_charge",
    "helix_beta_ratio",

    # Exposure / structure quality
    "sasa_per_res",
    "apolar_ratio",
    "mean_rsasa",
    "rama_fav",
    "rama_out",

    # Minimal length control
    "protein_length_aa",
]

selected_engineered_addons = [
    "canonical_similarity_max",
    "canonical_similarity_mean",
    "noncanonical_similarity_max",
    "noncanonical_similarity_mean",
    "noncanonical_minus_canonical_max",
    "noncanonical_minus_canonical_mean",
    "retron_group2_similarity_mean",
]

selected_handcrafted_v1 = [
    c for c in selected_handcrafted_v1
    if c in train_engineered.columns and pd.api.types.is_numeric_dtype(train_engineered[c])
]

selected_handcrafted_v2 = selected_handcrafted_v1 + [
    c for c in selected_engineered_addons
    if c in train_engineered.columns and pd.api.types.is_numeric_dtype(train_engineered[c])
]

selected_handcrafted_v2 = list(dict.fromkeys(selected_handcrafted_v2))

print("\nSelected v1:", len(selected_handcrafted_v1))
print(selected_handcrafted_v1)

print("\nSelected v2:", len(selected_handcrafted_v2))
print(selected_handcrafted_v2)


# ============================================================
# 4. LOFO model helpers
# ============================================================
def get_logistic_oof(df, feature_cols, c_value=0.01, model_name="logistic"):
    X = df[feature_cols].copy()

    model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            max_iter=5000,
            class_weight="balanced",
            penalty="l2",
            C=c_value,
            random_state=42
        ))
    ])

    logo = LeaveOneGroupOut()
    oof = np.zeros(len(df))
    fold_rows = []

    for fold, (tr_idx, va_idx) in enumerate(
        logo.split(X, y, groups=groups),
        start=1
    ):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y[tr_idx], y[va_idx]
        eff_va = eff[va_idx]
        fam = groups[va_idx][0]

        model.fit(X_tr, y_tr)
        pred = model.predict_proba(X_va)[:, 1]
        oof[va_idx] = pred

        if len(np.unique(y_va)) > 1:
            cls, pr_auc, w_spear = compute_cls(y_va, pred, eff_va)
        else:
            cls, pr_auc, w_spear = np.nan, np.nan, np.nan

        fold_rows.append({
            "model": model_name,
            "fold": fold,
            "holdout_family": fam,
            "n_valid": len(va_idx),
            "n_active_valid": int(y_va.sum()),
            "pr_auc": pr_auc,
            "weighted_spearman": w_spear,
            "cls": cls,
            "mean_score": float(np.mean(pred)),
            "median_score": float(np.median(pred)),
        })

    return oof, pd.DataFrame(fold_rows)


def get_embedding_pca_oof(n_components=3, c_value=0.1):
    emb_path = os.path.join(DATA_DIR, "esm2_embeddings.npz")

    if not os.path.exists(emb_path):
        raise FileNotFoundError(f"Missing embedding file: {emb_path}")

    emb_data = np.load(emb_path, allow_pickle=True)
    emb_dict = dict(zip(emb_data["names"], emb_data["embeddings"]))

    X_emb_full = np.vstack([emb_dict[name] for name in train[id_col]])
    X = pd.DataFrame(
        X_emb_full,
        columns=[f"esm2_{i}" for i in range(X_emb_full.shape[1])]
    )

    model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("pca", PCA(n_components=n_components, random_state=42)),
        ("clf", LogisticRegression(
            max_iter=5000,
            class_weight="balanced",
            penalty="l2",
            C=c_value,
            random_state=42
        ))
    ])

    logo = LeaveOneGroupOut()
    oof = np.zeros(len(train))
    fold_rows = []

    for fold, (tr_idx, va_idx) in enumerate(
        logo.split(X, y, groups=groups),
        start=1
    ):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y[tr_idx], y[va_idx]
        eff_va = eff[va_idx]
        fam = groups[va_idx][0]

        model.fit(X_tr, y_tr)
        pred = model.predict_proba(X_va)[:, 1]
        oof[va_idx] = pred

        if len(np.unique(y_va)) > 1:
            cls, pr_auc, w_spear = compute_cls(y_va, pred, eff_va)
        else:
            cls, pr_auc, w_spear = np.nan, np.nan, np.nan

        fold_rows.append({
            "model": f"embedding_pca{n_components}",
            "fold": fold,
            "holdout_family": fam,
            "n_valid": len(va_idx),
            "n_active_valid": int(y_va.sum()),
            "pr_auc": pr_auc,
            "weighted_spearman": w_spear,
            "cls": cls,
            "mean_score": float(np.mean(pred)),
            "median_score": float(np.median(pred)),
        })

    return oof, pd.DataFrame(fold_rows)


# ============================================================
# 5. Build base scores
# ============================================================
selected_v1_score, selected_v1_folds = get_logistic_oof(
    train_engineered,
    selected_handcrafted_v1,
    c_value=0.01,
    model_name="selected_v1_C0.01"
)

selected_v2_score, selected_v2_folds = get_logistic_oof(
    train_engineered,
    selected_handcrafted_v2,
    c_value=0.01,
    model_name="selected_v2_C0.01"
)

embedding_score, embedding_folds = get_embedding_pca_oof(
    n_components=3,
    c_value=0.1
)

# Search selected_v2 + embedding blend
base_blend_rows = []

for w_emb in np.arange(0.00, 1.01, 0.01):
    score = (1 - w_emb) * selected_v2_score + w_emb * embedding_score
    cls, pr_auc, w_spear = compute_cls(y, score, eff)

    base_blend_rows.append({
        "embedding_weight": round(float(w_emb), 2),
        "selected_v2_weight": round(float(1 - w_emb), 2),
        "pr_auc": pr_auc,
        "weighted_spearman": w_spear,
        "cls": cls,
    })

base_blend_df = (
    pd.DataFrame(base_blend_rows)
    .sort_values("cls", ascending=False)
    .reset_index(drop=True)
)

best_w_emb = float(base_blend_df.loc[0, "embedding_weight"])
base_score = (1 - best_w_emb) * selected_v2_score + best_w_emb * embedding_score

print("\n" + "=" * 80)
print("BASE SELECTED_V2 + EMBEDDING BLEND")
print("=" * 80)
display(base_blend_df.head(15))


# ============================================================
# 6. Build structure gates
# ============================================================
canonical_cols = [
    "foldseek_TM_HIV1",
    "foldseek_TM_MMLV",
    "foldseek_TM_MMLVPE",
]

canonical_cols = [c for c in canonical_cols if c in train_engineered.columns]

canonical_max = train_engineered[canonical_cols].max(axis=1).values
retron_tm = train_engineered["foldseek_TM_Retron"].values
group2_tm = train_engineered["foldseek_TM_Group2Intron"].values
telomerase_tm = train_engineered["foldseek_TM_Telomerase"].values

# LTR column exists in the original dataset if available.
if "foldseek_TM_LTRRetrotransposon" in train_engineered.columns:
    ltr_tm = train_engineered["foldseek_TM_LTRRetrotransposon"].values
else:
    ltr_tm = np.zeros(len(train_engineered))

noncanonical_max = np.maximum.reduce([retron_tm, group2_tm, telomerase_tm])

# Retron-like gate:
# high when Retron similarity is strong and stronger than canonical similarity.
retron_gate_raw = (
    0.70 * safe_zscore(retron_tm - canonical_max)
    + 0.30 * safe_zscore(retron_tm)
)

retron_gate = sigmoid(2.0 * retron_gate_raw) * rank01(retron_tm)

# LTR-like gate:
# high when LTR similarity is high.
ltr_gate_raw = (
    0.70 * safe_zscore(ltr_tm)
    + 0.30 * safe_zscore(ltr_tm - noncanonical_max)
)

ltr_gate = sigmoid(2.0 * ltr_gate_raw) * rank01(ltr_tm)

# Canonical-like confidence
canonical_gate_raw = (
    0.70 * safe_zscore(canonical_max - noncanonical_max)
    + 0.30 * safe_zscore(canonical_max)
)

canonical_gate = sigmoid(2.0 * canonical_gate_raw) * rank01(canonical_max)

gate_df = train[[id_col, group_col, target_col, eff_col]].copy()
gate_df["canonical_max"] = canonical_max
gate_df["retron_tm"] = retron_tm
gate_df["ltr_tm"] = ltr_tm
gate_df["noncanonical_max"] = noncanonical_max
gate_df["canonical_gate"] = canonical_gate
gate_df["retron_gate"] = retron_gate
gate_df["ltr_gate"] = ltr_gate

print("\n" + "=" * 80)
print("GATE SUMMARY BY FAMILY")
print("=" * 80)
display(
    gate_df.groupby(group_col)[
        ["canonical_gate", "retron_gate", "ltr_gate", "canonical_max", "retron_tm", "ltr_tm"]
    ].mean()
)


# ============================================================
# 7. Build FP-like penalty score from in-memory block diagnostics
# ============================================================
needed_block_cols = [
    "score_thermal_stability",
    "score_exposure_quality",
    "score_contacts_and_pocket",
    "score_active_site_geometry",
    "score_embedding_pca3",
    "score_global_structure",
    "score_canonical_retroviral_similarity",
    "score_electrostatics",
]

missing = [c for c in needed_block_cols if c not in block_diag.columns]
if missing:
    raise ValueError(f"Missing block score columns in diagnostic_df from Step 4-1: {missing}")

# Align block diagnostics to train order.
block_diag = train[[id_col]].merge(
    block_diag,
    on=id_col,
    how="left"
)

# Rank-normalize block scores
for col in needed_block_cols:
    block_diag[f"rank_{col}"] = rank01(block_diag[col].values)

fp_positive_side = (
    0.25 * block_diag["rank_score_thermal_stability"].values
    + 0.25 * block_diag["rank_score_exposure_quality"].values
    + 0.25 * block_diag["rank_score_contacts_and_pocket"].values
    + 0.25 * block_diag["rank_score_active_site_geometry"].values
)

fp_protective_side = (
    0.30 * block_diag["rank_score_embedding_pca3"].values
    + 0.30 * block_diag["rank_score_global_structure"].values
    + 0.25 * block_diag["rank_score_canonical_retroviral_similarity"].values
    + 0.15 * block_diag["rank_score_electrostatics"].values
)

fp_like_raw = fp_positive_side - fp_protective_side
fp_like_score = rank01(fp_like_raw)

# Apply FP penalty mostly where LTR/canonical-like false positives are common,
# and less strongly where Retron rescue is needed.
fp_penalty_gate = np.maximum(ltr_gate, canonical_gate) * (1 - 0.50 * retron_gate)

penalty_df = train[[id_col, group_col, target_col, eff_col]].copy()
penalty_df["fp_like_score"] = fp_like_score
penalty_df["fp_penalty_gate"] = fp_penalty_gate
penalty_df["fp_positive_side"] = fp_positive_side
penalty_df["fp_protective_side"] = fp_protective_side

print("\n" + "=" * 80)
print("FP-LIKE PENALTY SUMMARY BY FAMILY")
print("=" * 80)
display(
    penalty_df.groupby(group_col)[
        ["fp_like_score", "fp_penalty_gate", "fp_positive_side", "fp_protective_side"]
    ].mean()
)


# ============================================================
# 8. Structure-aware scoring sweep
# ============================================================
base_rank = rank01(base_score)
selected_v1_rank = rank01(selected_v1_score)
selected_v2_rank = rank01(selected_v2_score)
embedding_rank = rank01(embedding_score)

structure_rows = []
structure_score_store = {}

for alpha_retron in np.arange(0.00, 1.01, 0.05):
    # Retron rescue:
    # Move base score toward selected_v1 when retron_gate is high.
    retron_rescued = (
        (1 - alpha_retron * retron_gate) * base_rank
        + (alpha_retron * retron_gate) * selected_v1_rank
    )

    for beta_fp in np.arange(0.00, 0.51, 0.025):
        # FP penalty:
        # Subtract penalty in rank space, then re-rank.
        adjusted = retron_rescued - beta_fp * fp_like_score * fp_penalty_gate
        adjusted_rank = rank01(adjusted)

        cls, pr_auc, w_spear = compute_cls(y, adjusted_rank, eff)

        key = f"alpha_retron{alpha_retron:.2f}_beta_fp{beta_fp:.3f}"
        structure_score_store[key] = adjusted_rank

        structure_rows.append({
            "alpha_retron": round(float(alpha_retron), 2),
            "beta_fp": round(float(beta_fp), 3),
            "pr_auc": pr_auc,
            "weighted_spearman": w_spear,
            "cls": cls,
            "score_key": key,
        })

structure_sweep_df = (
    pd.DataFrame(structure_rows)
    .sort_values("cls", ascending=False)
    .reset_index(drop=True)
)

best_key = structure_sweep_df.loc[0, "score_key"]
best_structure_score = structure_score_store[best_key]

print("\n" + "=" * 80)
print("STRUCTURE-AWARE SWEEP SUMMARY")
print("=" * 80)
display(structure_sweep_df.head(30))


# ============================================================
# 9. Compare models
# ============================================================
structure_summary_rows = []

structure_summary_rows.append(evaluate_score("selected_v1_C0.01", selected_v1_score))
structure_summary_rows.append(evaluate_score("selected_v2_C0.01", selected_v2_score))
structure_summary_rows.append(evaluate_score("embedding_pca3", embedding_score))
structure_summary_rows.append(evaluate_score(
    f"base_selected_v2_plus_embedding_wemb{best_w_emb:.2f}",
    base_score
))
structure_summary_rows.append(evaluate_score("structure_aware_best", best_structure_score))

structure_summary_df = (
    pd.DataFrame(structure_summary_rows)
    .sort_values("cls", ascending=False)
    .reset_index(drop=True)
)

print("\n" + "=" * 80)
print("MODEL COMPARISON")
print("=" * 80)
display(structure_summary_df)

# ------------------------------------------------------------
# Safe family-level evaluator
# Works whether y / eff / groups are pandas Series or numpy arrays
# ------------------------------------------------------------
def evaluate_by_family(score, model_name):
    rows = []
    score = np.asarray(score, dtype=float)

    for fam in pd.unique(groups):
        idx = np.asarray(groups) == fam

        y_fam = np.asarray(y)[idx]
        eff_fam = np.asarray(eff)[idx]
        score_fam = score[idx]

        if len(np.unique(y_fam)) > 1:
            cls, pr_auc, w_spear = compute_cls(y_fam, score_fam, eff_fam)
        else:
            cls, pr_auc, w_spear = np.nan, np.nan, np.nan

        rows.append({
            "model": model_name,
            "family": fam,
            "n": int(idx.sum()),
            "n_active": int(y_fam.sum()),
            "pr_auc": pr_auc,
            "weighted_spearman": w_spear,
            "cls": cls,
            "mean_score": float(np.nanmean(score_fam)),
            "median_score": float(np.nanmedian(score_fam)),
        })

    return pd.DataFrame(rows)
    
# ============================================================
# 10. Family-level comparison
# ============================================================
structure_family_compare = pd.concat([
    evaluate_by_family(base_score, "base_selected_v2_plus_embedding"),
    evaluate_by_family(selected_v1_score, "selected_v1_C0.01"),
    evaluate_by_family(selected_v2_score, "selected_v2_C0.01"),
    evaluate_by_family(best_structure_score, "structure_aware_best"),
], ignore_index=True)

print("\n" + "=" * 80)
print("FAMILY-LEVEL COMPARISON")
print("=" * 80)
display(structure_family_compare.sort_values(["family", "model"]))


# ============================================================
# 11. Diagnostics for downstream targeted correction
# ============================================================
structure_diag = train[[id_col, group_col, target_col, eff_col]].copy()
structure_diag["selected_v1_score"] = selected_v1_score
structure_diag["selected_v2_score"] = selected_v2_score
structure_diag["embedding_score"] = embedding_score
structure_diag["base_score"] = base_score
structure_diag["structure_aware_score"] = best_structure_score
structure_diag["base_rank"] = base_rank
structure_diag["selected_v1_rank"] = selected_v1_rank
structure_diag["selected_v2_rank"] = selected_v2_rank
structure_diag["embedding_rank"] = embedding_rank
structure_diag["structure_aware_rank"] = rank01(best_structure_score)
structure_diag["retron_gate"] = retron_gate
structure_diag["ltr_gate"] = ltr_gate
structure_diag["canonical_gate"] = canonical_gate
structure_diag["fp_like_score"] = fp_like_score
structure_diag["fp_penalty_gate"] = fp_penalty_gate
structure_diag["fp_positive_side"] = fp_positive_side
structure_diag["fp_protective_side"] = fp_protective_side

# For compatibility with later cells that expect `diag`
diag = structure_diag.copy()

print("\n" + "=" * 80)
print("TOP 25 BY STRUCTURE-AWARE SCORE")
print("=" * 80)
display(
    structure_diag.sort_values("structure_aware_score", ascending=False).head(25)
)

print("\n" + "=" * 80)
print("TOP FALSE POSITIVES AFTER STRUCTURE-AWARE SCORING")
print("=" * 80)
display(
    structure_diag[structure_diag[target_col] == 0]
    .sort_values("structure_aware_score", ascending=False)
    .head(20)
)

print("\n" + "=" * 80)
print("BOTTOM FALSE NEGATIVES AFTER STRUCTURE-AWARE SCORING")
print("=" * 80)
display(
    structure_diag[structure_diag[target_col] == 1]
    .sort_values("structure_aware_score", ascending=True)
    .head(20)
)

print("\n" + "=" * 80)
print("RETRON DIAGNOSTICS")
print("=" * 80)
display(
    structure_diag[structure_diag[group_col] == "Retron"]
    .sort_values("structure_aware_score", ascending=False)
)

print("\n" + "=" * 80)
print("LTR DIAGNOSTICS")
print("=" * 80)
display(
    structure_diag[structure_diag[group_col] == "LTR_Retrotransposon"]
    .sort_values("structure_aware_score", ascending=False)
)

print("\nStep 4-2 outputs ready.")
print("structure_diag:", structure_diag.shape)
print("diag:", diag.shape)
print("Structure-aware score columns:")
print([c for c in structure_diag.columns if "score" in c])

STRUCTURE-AWARE INPUT
train: (57, 71)


rt_family
Retroviral             18
Retron                 12
LTR_Retrotransposon    11
Group_II_Intron         5
CRISPR-associated       5
Other                   5
Unclassified            1
Name: count, dtype: int64


Block diagnostic input: (57, 17)
Block score columns:
['score_global_structure', 'score_canonical_retroviral_similarity', 'score_noncanonical_similarity', 'score_active_site_geometry', 'score_thermal_stability', 'score_physicochemical', 'score_contacts_and_pocket', 'score_exposure_quality', 'score_electrostatics', 'score_embedding_pca3', 'score_full_engineered', 'score_full_two_branch_best', 'score_best_block_blend']

Selected v1: 27
['foldseek_best_TM', 'foldseek_best_LDDT', 'foldseek_best_fident', 'foldseek_TM_HIV1', 'foldseek_TM_MMLV', 'foldseek_TM_MMLVPE', 'foldseek_TM_Retron', 'foldseek_TM_Group2Intron', 'foldseek_TM_Telomerase', 'triad_found_bin', 'D1_D2_dist', 'D2_D3_dist', 'triad_best_rmsd', 'hairpin_pass', 'hairpin_confidence', 'perplexity', 'instability_index', 'gravy', 'camsol', 'net_charge', 'helix_beta_ratio', 'sasa_per_res', 'apolar_ratio', 'mean_rsasa', 'rama_fav', 'rama_out', 'protein_length_aa']

Selected v2: 34
['foldseek_best_TM', 'foldseek_best_LDDT', 'foldseek_bes

,embedding_weight,selected_v2_weight,pr_auc,weighted_spearman,cls
0,0.23,0.77,0.658568,0.682913,0.670520
1,0.22,0.78,0.657687,0.679297,0.668317
2,0.24,0.76,0.654009,0.680645,0.667061
3,0.21,0.79,0.656805,0.676957,0.666729
4,0.27,0.73,0.647233,0.685636,0.665881
5,0.25,0.75,0.647365,0.683448,0.664917
6,0.26,0.74,0.647365,0.683427,0.664907
7,0.19,0.81,0.655262,0.673394,0.664204
8,0.20,0.80,0.656083,0.671492,0.663698
9,0.28,0.72,0.639297,0.689018,0.663227



GATE SUMMARY BY FAMILY


...


RETRON DIAGNOSTICS


,rt_name,rt_family,active,pe_efficiency_pct,selected_v1_score,selected_v2_score,embedding_score,base_score,structure_aware_score,base_rank,...,selected_v2_rank,embedding_rank,structure_aware_rank,retron_gate,ltr_gate,canonical_gate,fp_like_score,fp_penalty_gate,fp_positive_side,fp_protective_side
36,Vp96-RT,Retron,1,8.0,0.500691,0.459443,0.199359,0.399623,0.500000,0.428571,...,0.589286,0.196429,0.500000,0.966677,0.098206,0.053593,0.339286,0.050739,0.254464,0.362500
30,Retron86-RT,Retron,0,0.0,0.492326,0.444356,0.135255,0.373263,0.428571,0.285714,...,0.482143,0.053571,0.428571,0.872921,0.108573,0.030518,0.428571,0.061185,0.383929,0.433036
32,SEN1-RT,Retron,0,0.0,0.468936,0.432459,0.228313,0.385505,0.392857,0.357143,...,0.410714,0.267857,0.392857,0.923172,0.129374,0.062290,0.392857,0.069657,0.290179,0.380357
37,"retronE,coli-RT",Retron,0,0.0,0.479173,0.435593,0.135763,0.366632,0.321429,0.267857,...,0.428571,0.071429,0.321429,0.878754,0.128023,0.068132,0.267857,0.071773,0.321429,0.459821
35,Vc95-RT,Retron,1,1.5,0.419099,0.384201,0.198101,0.341398,0.250000,0.232143,...,0.267857,0.178571,0.250000,0.947381,0.116255,0.059989,0.160714,0.061186,0.165179,0.364286
33,Sau1-RT,Retron,0,0.0,0.333036,0.308103,0.270562,0.299468,0.214286,0.196429,...,0.178571,0.392857,0.214286,0.752169,0.087791,0.074702,0.125000,0.054774,0.066964,0.288393
27,Ec67-RT,Retron,0,0.0,0.355245,0.316678,0.175137,0.284123,0.160714,0.142857,...,0.196429,0.142857,0.160714,0.671793,0.109843,0.018267,0.875000,0.072947,0.375000,0.160714
26,Ec48-RT,Retron,1,1.5,0.335762,0.305279,0.173280,0.274919,0.107143,0.089286,...,0.160714,0.125000,0.107143,0.775776,0.031423,0.059725,0.321429,0.036558,0.138393,0.251786
31,Rs-RT,Retron,1,2.5,0.337952,0.286079,0.144313,0.253472,0.071429,0.053571,...,0.125000,0.089286,0.071429,0.697153,0.487618,0.006381,0.857143,0.317646,0.406250,0.193750
28,Mx65-RT,Retron,0,0.0,0.317679,0.282411,0.202050,0.263928,0.053571,0.071429,...,0.089286,0.214286,0.053571,0.858770,0.001675,0.014901,0.214286,0.008503,0.080357,0.234821



LTR DIAGNOSTICS


,rt_name,rt_family,active,pe_efficiency_pct,selected_v1_score,selected_v2_score,embedding_score,base_score,structure_aware_score,base_rank,...,selected_v2_rank,embedding_rank,structure_aware_rank,retron_gate,ltr_gate,canonical_gate,fp_like_score,fp_penalty_gate,fp_positive_side,fp_protective_side
16,Ngaro-RT,LTR_Retrotransposon,0,0.0,0.511804,0.517481,0.769010,0.575333,0.857143,0.857143,...,0.750000,1.000000,0.857143,0.046912,0.663911,0.453245,0.839286,0.648338,0.812500,0.609821
14,Gypsy-RT,LTR_Retrotransposon,0,0.0,0.550098,0.580715,0.503136,0.562871,0.839286,0.839286,...,0.857143,0.750000,0.839286,0.013739,0.323502,0.677650,0.678571,0.672995,0.839286,0.719643
17,Tf1-RT,LTR_Retrotransposon,1,34.0,0.505642,0.530465,0.562897,0.537924,0.821429,0.821429,...,0.785714,0.964286,0.821429,0.201035,0.457751,0.570749,0.500000,0.513378,0.776786,0.791071
19,Ty3-RT,LTR_Retrotransposon,1,9.0,0.502626,0.538005,0.520085,0.533883,0.803571,0.803571,...,0.839286,0.839286,0.803571,0.012558,0.918828,0.607007,0.571429,0.913058,0.718750,0.701786
10,Bel-RT,LTR_Retrotransposon,0,0.0,0.519138,0.537998,0.503830,0.530140,0.785714,0.785714,...,0.821429,0.767857,0.785714,0.013921,0.008900,0.474792,0.910714,0.471488,0.915179,0.620536
20,Tyrosinerecomb-RT,LTR_Retrotransposon,0,0.0,0.517316,0.534138,0.408113,0.505152,0.750000,0.750000,...,0.803571,0.607143,0.750000,0.046576,0.821486,0.529682,0.982143,0.802355,0.928571,0.514286
13,Gate-RT,LTR_Retrotransposon,0,0.0,0.479970,0.498796,0.486934,0.496068,0.732143,0.732143,...,0.714286,0.696429,0.732143,0.000000,0.052475,0.433525,1.000000,0.433525,0.879464,0.439286
18,Ty1B-RT,LTR_Retrotransposon,0,0.0,0.392907,0.414311,0.736565,0.488429,0.696429,0.696429,...,0.357143,0.982143,0.696429,0.007566,0.031668,0.414535,0.410714,0.412967,0.482143,0.553571
15,Line1-RT,LTR_Retrotransposon,0,0.0,0.400384,0.390689,0.508628,0.417815,0.410714,0.482143,...,0.285714,0.803571,0.410714,0.437577,0.006843,0.155008,0.946429,0.121094,0.799107,0.449107
11,CHLRE,LTR_Retrotransposon,0,0.0,0.363528,0.340446,0.095210,0.284042,0.125000,0.125000,...,0.214286,0.017857,0.125000,0.244322,0.226285,0.017094,0.964286,0.198642,0.566964,0.200000



Step 4-2 outputs ready.
structure_diag: (57, 21)
diag: (57, 21)
Structure-aware score columns:
['selected_v1_score', 'selected_v2_score', 'embedding_score', 'base_score', 'structure_aware_score', 'fp_like_score']


In [7]:
# =========================================
# Step 4-3. Targeted Correction Experiment
# In-memory version for final submission notebook
#
# Uses:
#   - diagnostic_df from Step 4-1
#   - structure_diag from Step 4-2
#
# Goal:
#   Build:
#   1. Retroviral FP penalty
#   2. LTR active correction
#   3. Retron active rescue
#
# Final score:
#   final_score =
#       base_score
#       - a * retroviral_fp_penalty
#       + b * ltr_active_correction
#       + c * retron_active_rescue
# =========================================

# ------------------------------------------------------------
# 0. Compatibility setup
# ------------------------------------------------------------
if "COMP_DIR" not in globals():
    if "DATA_DIR" in globals():
        COMP_DIR = DATA_DIR
    else:
        COMP_DIR = "/kaggle/input/competitions/retroviral-challenge-predict"

DATA_DIR = COMP_DIR

if "train_df" in globals():
    train = train_df.copy()
else:
    train = pd.read_csv(f"{DATA_DIR}/train.csv")

id_col = ID_COL if "ID_COL" in globals() else "rt_name"
family_col = FAMILY_COL if "FAMILY_COL" in globals() else "rt_family"
group_col = family_col
target_col = TARGET_COL if "TARGET_COL" in globals() else "active"
eff_col = EFF_COL if "EFF_COL" in globals() else "pe_efficiency_pct"

y = train[target_col].astype(int).values
eff = train[eff_col].astype(float).values
groups = train[family_col].values

print("=" * 80)
print("TARGETED CORRECTION INPUT")
print("=" * 80)
print("train:", train.shape)
display(train[family_col].value_counts())

# Required previous-cell objects
if "diagnostic_df" not in globals():
    raise ValueError(
        "diagnostic_df not found. Run Step 4-1 block score generation first."
    )

if "structure_diag" not in globals():
    raise ValueError(
        "structure_diag not found. Run Step 4-2 structure-aware scoring first."
    )


# =========================================
# 1. Helper functions
# =========================================
def mean_existing(df, cols):
    valid = [c for c in cols if c in df.columns]
    if len(valid) == 0:
        return np.zeros(len(df))
    return df[valid].mean(axis=1).values


def rank_col(df, col, invert=False):
    if col not in df.columns:
        print(f"Warning: missing column {col}. Using zeros.")
        out = np.zeros(len(df))
    else:
        out = rank01(df[col].values)

    if invert:
        out = 1 - out

    return out


# =========================================
# 2. Add engineered similarity features
# =========================================
def add_simple_rt_similarity_features(df):
    df = df.copy()

    canonical_cols = [
        "foldseek_TM_HIV1",
        "foldseek_TM_MMLV",
        "foldseek_TM_MMLVPE",
    ]

    noncanonical_cols = [
        "foldseek_TM_Retron",
        "foldseek_TM_Group2Intron",
        "foldseek_TM_Telomerase",
    ]

    canonical_cols = [c for c in canonical_cols if c in df.columns]
    noncanonical_cols = [c for c in noncanonical_cols if c in df.columns]

    if len(canonical_cols) > 0:
        df["canonical_similarity_max"] = df[canonical_cols].max(axis=1)
        df["canonical_similarity_mean"] = df[canonical_cols].mean(axis=1)

    if len(noncanonical_cols) > 0:
        df["noncanonical_similarity_max"] = df[noncanonical_cols].max(axis=1)
        df["noncanonical_similarity_mean"] = df[noncanonical_cols].mean(axis=1)

    if "canonical_similarity_max" in df.columns and "noncanonical_similarity_max" in df.columns:
        df["noncanonical_minus_canonical_max"] = (
            df["noncanonical_similarity_max"] - df["canonical_similarity_max"]
        )

    if "canonical_similarity_mean" in df.columns and "noncanonical_similarity_mean" in df.columns:
        df["noncanonical_minus_canonical_mean"] = (
            df["noncanonical_similarity_mean"] - df["noncanonical_similarity_mean"]
        )

    # Fix: overwrite the line above with the correct formula
    if "canonical_similarity_mean" in df.columns and "noncanonical_similarity_mean" in df.columns:
        df["noncanonical_minus_canonical_mean"] = (
            df["noncanonical_similarity_mean"] - df["canonical_similarity_mean"]
        )

    if "foldseek_TM_Retron" in df.columns and "canonical_similarity_max" in df.columns:
        df["retron_minus_canonical_max"] = (
            df["foldseek_TM_Retron"] - df["canonical_similarity_max"]
        )

    if "foldseek_TM_Retron" in df.columns and "canonical_similarity_mean" in df.columns:
        df["retron_minus_canonical_mean"] = (
            df["foldseek_TM_Retron"] - df["canonical_similarity_mean"]
        )

    if "foldseek_TM_Retron" in df.columns and "foldseek_TM_Group2Intron" in df.columns:
        df["retron_group2_similarity_mean"] = df[
            ["foldseek_TM_Retron", "foldseek_TM_Group2Intron"]
        ].mean(axis=1)

    return df


train_eng = add_simple_rt_similarity_features(train)


# =========================================
# 3. Build diagnostic table in memory
# =========================================
diag = train_eng[[id_col, family_col, target_col, eff_col]].copy()

# Add raw numeric feature columns
numeric_cols = [
    c for c in train_eng.columns
    if c not in [id_col, "sequence", target_col, eff_col, family_col, "yxdd_seq"]
    and pd.api.types.is_numeric_dtype(train_eng[c])
]

for c in numeric_cols:
    diag[c] = train_eng[c]

# Add block score columns from Step 4-1 diagnostic_df
block_score_cols = [c for c in diagnostic_df.columns if c.startswith("score_")]

if len(block_score_cols) == 0:
    raise ValueError(
        "No block score columns found in diagnostic_df. "
        "Step 4-1 may not have run correctly."
    )

diag = diag.merge(
    diagnostic_df[[id_col] + block_score_cols],
    on=id_col,
    how="left"
)

# Add structure-aware columns from Step 4-2 structure_diag
structure_extra_cols = [
    c for c in [
        "selected_v1_score",
        "selected_v2_score",
        "embedding_score",
        "base_score",
        "structure_aware_score",
        "retron_gate",
        "ltr_gate",
        "canonical_gate",
        "fp_like_score",
        "fp_penalty_gate",
    ]
    if c in structure_diag.columns
]

missing_structure_cols = [
    c for c in [
        "base_score",
        "structure_aware_score",
        "retron_gate",
        "ltr_gate",
        "canonical_gate",
        "fp_like_score",
        "fp_penalty_gate",
    ]
    if c not in structure_diag.columns
]

if missing_structure_cols:
    raise ValueError(
        f"Missing required structure-aware columns in structure_diag: {missing_structure_cols}"
    )

diag = diag.merge(
    structure_diag[[id_col] + structure_extra_cols],
    on=id_col,
    how="left"
)

if "structure_aware_score" not in diag.columns:
    raise ValueError("structure_aware_score not found.")

if "base_score" not in diag.columns:
    raise ValueError("base_score not found.")

diag["base_selected_embedding_score"] = diag["base_score"]
diag["base_structure_aware_score"] = diag["structure_aware_score"]

print("\n" + "=" * 80)
print("DIAGNOSTIC TABLE")
print("=" * 80)
print(diag.shape)
print("Available score columns:")
print([c for c in diag.columns if "score" in c][:60])


# =========================================
# 4. Build gates
# =========================================
for gate_col in ["canonical_gate", "ltr_gate", "retron_gate"]:
    if gate_col not in diag.columns:
        raise ValueError(f"{gate_col} is missing. Run Step 4-2 first.")

canonical_gate = diag["canonical_gate"].fillna(0).values
ltr_gate = diag["ltr_gate"].fillna(0).values
retron_gate = diag["retron_gate"].fillna(0).values

retroviral_like_gate = canonical_gate * (1 - 0.50 * retron_gate)
ltr_like_gate = ltr_gate * (1 - 0.50 * retron_gate)
retron_like_gate = retron_gate

diag["retroviral_like_gate"] = retroviral_like_gate
diag["ltr_like_gate"] = ltr_like_gate
diag["retron_like_gate"] = retron_like_gate

print("\n" + "=" * 80)
print("GATE SUMMARY BY FAMILY")
print("=" * 80)
display(
    diag.groupby(family_col)[
        [
            "canonical_gate",
            "ltr_gate",
            "retron_gate",
            "retroviral_like_gate",
            "ltr_like_gate",
            "retron_like_gate",
        ]
    ].mean()
)


# =========================================
# 5. Retroviral FP penalty
# =========================================
retro_fp_risk = (
    0.20 * rank_col(diag, "perplexity")
    + 0.20 * rank_col(diag, "pocket_mean_rsasa")
    + 0.15 * rank_col(diag, "motif12_mean_pot")
    + 0.15 * rank_col(diag, "t40_raw")
    + 0.10 * rank_col(diag, "mean_rsasa")
    + 0.10 * rank_col(diag, "instability_index")
    + 0.10 * rank_col(diag, "triad_best_rmsd")
)

retro_fp_protective = (
    0.25 * rank_col(diag, "yxdd_std_hydrophobicity")
    + 0.20 * rank_col(diag, "log_likelihood")
    + 0.15 * rank_col(diag, "hbonds_per_res")
    + 0.15 * rank_col(diag, "embedding_score")
    + 0.15 * rank_col(diag, "selected_v1_score")
    + 0.10 * rank_col(diag, "score_full_engineered")
)

retroviral_fp_penalty_raw = retro_fp_risk - retro_fp_protective
retroviral_fp_penalty = rank01(retroviral_fp_penalty_raw) * retroviral_like_gate

diag["retro_fp_risk"] = retro_fp_risk
diag["retro_fp_protective"] = retro_fp_protective
diag["retroviral_fp_penalty"] = retroviral_fp_penalty


# =========================================
# 6. LTR active correction
# =========================================
ltr_active_side = (
    0.18 * rank_col(diag, "ltr_gate")
    + 0.16 * rank_col(diag, "pocket_hydrophobic_per_res")
    + 0.14 * rank_col(diag, "score_canonical_retroviral_similarity")
    + 0.12 * rank_col(diag, "foldseek_TM_MMLVPE")
    + 0.10 * rank_col(diag, "canonical_similarity_mean")
    + 0.10 * rank_col(diag, "triad_found_bin")
    + 0.08 * rank_col(diag, "score_electrostatics")
    + 0.06 * rank_col(diag, "mean_rsasa")
    + 0.06 * rank_col(diag, "pocket_mean_rsasa")
)

ltr_fp_side = (
    0.16 * rank_col(diag, "lpqg_sp_3A_mean_pot")
    + 0.14 * rank_col(diag, "yxdd_std_hydrophobicity")
    + 0.12 * rank_col(diag, "hbonds_per_res")
    + 0.12 * rank_col(diag, "gravy")
    + 0.12 * rank_col(diag, "foldseek_TM_Telomerase")
    + 0.12 * rank_col(diag, "fp_like_score")
    + 0.10 * rank_col(diag, "rama_out")
    + 0.08 * rank_col(diag, "yxdd_5A_mean_pot")
    + 0.04 * rank_col(diag, "score_exposure_quality")
)

ltr_active_correction_raw = ltr_active_side - ltr_fp_side
ltr_active_correction = rank01(ltr_active_correction_raw) * ltr_like_gate

diag["ltr_active_side"] = ltr_active_side
diag["ltr_fp_side"] = ltr_fp_side
diag["ltr_active_correction"] = ltr_active_correction


# =========================================
# 7. Retron active rescue
# =========================================
retron_active_side = (
    0.16 * rank_col(diag, "yxdd_hydrophobic_fraction")
    + 0.14 * rank_col(diag, "hydrophobic_per_res")
    + 0.12 * rank_col(diag, "foldseek_best_TM")
    + 0.10 * rank_col(diag, "rama_fav")
    + 0.10 * rank_col(diag, "yxdd_mean_hydrophobicity")
    + 0.10 * rank_col(diag, "score_contacts_and_pocket")
    + 0.08 * rank_col(diag, "hairpin_pass")
    + 0.08 * rank_col(diag, "noncanonical_similarity_max")
    + 0.06 * rank_col(diag, "retron_group2_similarity_mean")
    + 0.06 * rank_col(diag, "foldseek_best_LDDT")
)

retron_inactive_side = (
    0.12 * rank_col(diag, "hairpin_confidence")
    + 0.12 * rank_col(diag, "thumb_mean_pot")
    + 0.10 * rank_col(diag, "t60_raw")
    + 0.10 * rank_col(diag, "t50_raw")
    + 0.08 * rank_col(diag, "t45_raw")
    + 0.08 * rank_col(diag, "t55_raw")
    + 0.08 * rank_col(diag, "t40_raw")
    + 0.08 * rank_col(diag, "overall_mean_pot")
    + 0.08 * rank_col(diag, "net_charge")
    + 0.06 * rank_col(diag, "fingers_mean_pot")
    + 0.05 * rank_col(diag, "pocket_hbonds_per_res")
    + 0.05 * rank_col(diag, "protein_length_aa")
)

retron_active_rescue_raw = retron_active_side - retron_inactive_side
retron_active_rescue = rank01(retron_active_rescue_raw) * retron_like_gate

diag["retron_active_side"] = retron_active_side
diag["retron_inactive_side"] = retron_inactive_side
diag["retron_active_rescue"] = retron_active_rescue


# =========================================
# 8. Inspect correction scores
# =========================================
print("\n" + "=" * 80)
print("CORRECTION SCORE SUMMARY BY FAMILY")
print("=" * 80)
display(
    diag.groupby(family_col)[
        [
            "retroviral_fp_penalty",
            "ltr_active_correction",
            "retron_active_rescue",
            "retro_fp_risk",
            "retro_fp_protective",
            "ltr_active_side",
            "ltr_fp_side",
            "retron_active_side",
            "retron_inactive_side",
        ]
    ].mean()
)


# =========================================
# 9. Sweep correction weights
# =========================================
base_candidates = {
    "base_selected_embedding": rank01(diag["base_selected_embedding_score"].values),
    "base_structure_aware": rank01(diag["base_structure_aware_score"].values),
}

retro_pen = diag["retroviral_fp_penalty"].values
ltr_corr = diag["ltr_active_correction"].values
retron_corr = diag["retron_active_rescue"].values

targeted_sweep_rows = []
targeted_score_store = {}

a_values = np.round(np.arange(0.00, 0.151, 0.01), 3)
b_values = np.round(np.arange(0.00, 0.151, 0.01), 3)
c_values = np.round(np.arange(0.00, 0.151, 0.01), 3)

for base_name, base_score in base_candidates.items():
    for a in a_values:
        for b in b_values:
            for c in c_values:
                raw_score = (
                    base_score
                    - a * retro_pen
                    + b * ltr_corr
                    + c * retron_corr
                )

                final_score = rank01(raw_score)

                cls, pr_auc, w_spear = compute_cls(y, final_score, eff)

                key = f"{base_name}_a{a:.3f}_b{b:.3f}_c{c:.3f}"
                targeted_score_store[key] = final_score

                targeted_sweep_rows.append({
                    "base_name": base_name,
                    "a_retroviral_fp_penalty": a,
                    "b_ltr_active_correction": b,
                    "c_retron_active_rescue": c,
                    "pr_auc": pr_auc,
                    "weighted_spearman": w_spear,
                    "cls": cls,
                    "score_key": key,
                })

targeted_sweep_df = (
    pd.DataFrame(targeted_sweep_rows)
    .sort_values("cls", ascending=False)
    .reset_index(drop=True)
)

best_targeted_row = targeted_sweep_df.iloc[0]
best_targeted_key = best_targeted_row["score_key"]
best_targeted_score = targeted_score_store[best_targeted_key]

print("\n" + "=" * 80)
print("TARGETED CORRECTION SWEEP SUMMARY")
print("=" * 80)
display(targeted_sweep_df.head(40))


# =========================================
# 10. Final comparison
# =========================================
targeted_summary_rows = []

targeted_summary_rows.append(evaluate_score(
    "base_selected_embedding",
    base_candidates["base_selected_embedding"]
))

targeted_summary_rows.append(evaluate_score(
    "base_structure_aware",
    base_candidates["base_structure_aware"]
))

targeted_summary_rows.append(evaluate_score(
    "targeted_correction_best",
    best_targeted_score
))

targeted_summary_df = (
    pd.DataFrame(targeted_summary_rows)
    .sort_values("cls", ascending=False)
    .reset_index(drop=True)
)

print("\n" + "=" * 80)
print("MODEL COMPARISON")
print("=" * 80)
display(targeted_summary_df)

print("\nBest correction setting:")
display(best_targeted_row.to_frame().T)


# =========================================
# 11. Family-level comparison
# =========================================
targeted_family_compare = pd.concat([
    evaluate_by_family(base_candidates["base_selected_embedding"], "base_selected_embedding"),
    evaluate_by_family(base_candidates["base_structure_aware"], "base_structure_aware"),
    evaluate_by_family(best_targeted_score, "targeted_correction_best"),
], ignore_index=True)

print("\n" + "=" * 80)
print("FAMILY-LEVEL COMPARISON")
print("=" * 80)
display(targeted_family_compare.sort_values(["family", "model"]))


# =========================================
# 12. Diagnostics for downstream nested validation
# =========================================
diag["targeted_correction_score"] = best_targeted_score
diag["targeted_correction_rank"] = rank01(best_targeted_score)

# Explicit alias for next cell
targeted_diag = diag.copy()

print("\n" + "=" * 80)
print("TOP 25 BY TARGETED CORRECTION SCORE")
print("=" * 80)
display(
    targeted_diag.sort_values("targeted_correction_score", ascending=False)
    [[
        id_col, family_col, target_col, eff_col,
        "targeted_correction_score",
        "base_structure_aware_score",
        "retroviral_fp_penalty",
        "ltr_active_correction",
        "retron_active_rescue",
    ]]
    .head(25)
)

print("\n" + "=" * 80)
print("TOP FALSE POSITIVES AFTER TARGETED CORRECTION")
print("=" * 80)
display(
    targeted_diag[targeted_diag[target_col] == 0]
    .sort_values("targeted_correction_score", ascending=False)
    [[
        id_col, family_col, target_col, eff_col,
        "targeted_correction_score",
        "base_structure_aware_score",
        "retroviral_fp_penalty",
        "ltr_active_correction",
        "retron_active_rescue",
    ]]
    .head(20)
)

print("\n" + "=" * 80)
print("BOTTOM FALSE NEGATIVES AFTER TARGETED CORRECTION")
print("=" * 80)
display(
    targeted_diag[targeted_diag[target_col] == 1]
    .sort_values("targeted_correction_score", ascending=True)
    [[
        id_col, family_col, target_col, eff_col,
        "targeted_correction_score",
        "base_structure_aware_score",
        "retroviral_fp_penalty",
        "ltr_active_correction",
        "retron_active_rescue",
    ]]
    .head(20)
)

print("\nStep 4-3 outputs ready.")
print("targeted_diag:", targeted_diag.shape)
print("Best targeted correction setting:")
display(best_targeted_row.to_frame().T)

TARGETED CORRECTION INPUT
train: (57, 71)


rt_family
Retroviral             18
Retron                 12
LTR_Retrotransposon    11
Group_II_Intron         5
CRISPR-associated       5
Other                   5
Unclassified            1
Name: count, dtype: int64


DIAGNOSTIC TABLE
(57, 103)
Available score columns:
['score_global_structure', 'score_canonical_retroviral_similarity', 'score_noncanonical_similarity', 'score_active_site_geometry', 'score_thermal_stability', 'score_physicochemical', 'score_contacts_and_pocket', 'score_exposure_quality', 'score_electrostatics', 'score_embedding_pca3', 'score_full_engineered', 'score_full_two_branch_best', 'score_best_block_blend', 'selected_v1_score', 'selected_v2_score', 'embedding_score', 'base_score', 'structure_aware_score', 'fp_like_score', 'base_selected_embedding_score', 'base_structure_aware_score']

GATE SUMMARY BY FAMILY


,canonical_gate,ltr_gate,retron_gate,retroviral_like_gate,ltr_like_gate,retron_like_gate
rt_family,,,,,,
CRISPR-associated,0.036745,0.201811,0.597036,0.025551,0.135782,0.597036
Group_II_Intron,0.022653,0.346638,0.582483,0.016270,0.241186,0.582483
LTR_Retrotransposon,0.394410,0.320257,0.115393,0.382563,0.309403,0.115393
Other,0.137272,0.066503,0.353014,0.128972,0.062043,0.353014
Retron,0.044070,0.111519,0.786086,0.025466,0.067002,0.786086
Retroviral,0.753121,0.603309,0.055326,0.731305,0.590211,0.055326
Unclassified,0.224152,0.102751,0.113491,0.211432,0.096921,0.113491



CORRECTION SCORE SUMMARY BY FAMILY


...

,rt_name,rt_family,active,pe_efficiency_pct,targeted_correction_score,base_structure_aware_score,retroviral_fp_penalty,ltr_active_correction,retron_active_rescue
42,FENV1-RT,Retroviral,0,0.0,0.892857,0.910714,0.092119,0.281612,0.017263
16,Ngaro-RT,LTR_Retrotransposon,0,0.0,0.857143,0.857143,0.150173,0.196817,0.026807
14,Gypsy-RT,LTR_Retrotransposon,0,0.0,0.803571,0.839286,0.240356,0.206537,0.010059
10,Bel-RT,LTR_Retrotransposon,0,0.0,0.785714,0.785714,0.101033,0.005208,0.002735
20,Tyrosinerecomb-RT,LTR_Retrotransposon,0,0.0,0.767857,0.750000,0.314103,0.444161,0.020793
54,WDSV-RT,Retroviral,0,0.0,0.750000,0.767857,0.556583,0.347640,0.151245
13,Gate-RT,LTR_Retrotransposon,0,0.0,0.714286,0.732143,0.216762,0.027174,0.000000
22,CaMV-RT,Other,0,0.0,0.678571,0.714286,0.456652,0.212141,0.020168
18,Ty1B-RT,LTR_Retrotransposon,0,0.0,0.660714,0.696429,0.228607,0.018027,0.003648
2,FuRT-Cas1-RT,CRISPR-associated,0,0.0,0.642857,0.607143,0.010961,0.099396,0.603925



BOTTOM FALSE NEGATIVES AFTER TARGETED CORRECTION


,rt_name,rt_family,active,pe_efficiency_pct,targeted_correction_score,base_structure_aware_score,retroviral_fp_penalty,ltr_active_correction,retron_active_rescue
29,Ne144-RT,Retron,1,0.5,0.035714,0.035714,0.039401,0.011538,0.238872
31,Rs-RT,Retron,1,2.5,0.089286,0.071429,0.003117,0.079412,0.697153
26,Ec48-RT,Retron,1,1.5,0.125000,0.107143,0.017626,0.001374,0.540272
35,Vc95-RT,Retron,1,1.5,0.267857,0.250000,0.023116,0.022945,0.795123
48,MMTV-RT,Retroviral,1,19.0,0.446429,0.446429,0.355857,0.439734,0.023143
53,SRV2-RT,Retroviral,1,3.5,0.464286,0.464286,0.419069,0.553898,0.001333
49,MPMV-RT,Retroviral,1,7.0,0.517857,0.517857,0.361501,0.511728,0.016016
36,Vp96-RT,Retron,1,8.0,0.553571,0.500000,0.007911,0.013591,0.897629
5,Er-RT,Group_II_Intron,1,12.5,0.607143,0.571429,0.000427,0.046695,0.503984
7,Gs-RT,Group_II_Intron,1,1.0,0.625000,0.625000,0.002291,0.031596,0.224621



Step 4-3 outputs ready.
targeted_diag: (57, 117)
Best targeted correction setting:


,base_name,a_retroviral_fp_penalty,b_ltr_active_correction,c_retron_active_rescue,pr_auc,weighted_spearman,cls,score_key
0,base_structure_aware,0.15,0.12,0.07,0.692215,0.686964,0.68958,base_structure_aware_a0.150_b0.120_c0.070


In [8]:
# =========================================
# Step 4-4. Nested validation for targeted corrections
# Final submission version
#
# Uses:
#   - targeted_diag from Step 4-3
#
# Output:
#   - submission.csv
#
# Important:
#   This cell does NOT read/write intermediate CSV files.
#   It only writes the final submission.csv.
# =========================================

# ------------------------------------------------------------
# 0. Compatibility setup
# ------------------------------------------------------------
if "train_df" in globals():
    train = train_df.copy()
else:
    train = pd.read_csv(f"{DATA_DIR}/train.csv")

id_col = ID_COL if "ID_COL" in globals() else "rt_name"
family_col = FAMILY_COL if "FAMILY_COL" in globals() else "rt_family"
group_col = family_col
target_col = TARGET_COL if "TARGET_COL" in globals() else "active"
eff_col = EFF_COL if "EFF_COL" in globals() else "pe_efficiency_pct"

y = train[target_col].astype(int).values
eff = train[eff_col].astype(float).values
groups = train[family_col].values

if "targeted_diag" not in globals():
    raise ValueError(
        "targeted_diag not found. Run Step 4-3 targeted correction first."
    )

diag = targeted_diag.copy()

required_cols = [
    "base_selected_embedding_score",
    "base_structure_aware_score",
    "retroviral_fp_penalty",
    "ltr_active_correction",
    "retron_active_rescue",
]

missing = [c for c in required_cols if c not in diag.columns]
if missing:
    raise ValueError(f"Missing required columns in targeted_diag: {missing}")

print("=" * 80)
print("NESTED VALIDATION INPUT")
print("=" * 80)
print("diag:", diag.shape)
print("required columns OK")


# =========================================
# 1. Safe local evaluators
# =========================================
def evaluate_score_local(name, score):
    cls, pr_auc, w_spear = compute_cls(y, score, eff)

    return {
        "model": name,
        "pr_auc": pr_auc,
        "weighted_spearman": w_spear,
        "cls": cls,
    }


def evaluate_by_family_local(score, model_name):
    rows = []
    score = np.asarray(score, dtype=float)

    for fam in pd.unique(groups):
        idx = np.asarray(groups) == fam

        y_fam = np.asarray(y)[idx]
        eff_fam = np.asarray(eff)[idx]
        score_fam = score[idx]

        if len(np.unique(y_fam)) > 1:
            cls, pr_auc, w_spear = compute_cls(y_fam, score_fam, eff_fam)
        else:
            cls, pr_auc, w_spear = np.nan, np.nan, np.nan

        rows.append({
            "model": model_name,
            "family": fam,
            "n": int(idx.sum()),
            "n_active": int(y_fam.sum()),
            "pr_auc": pr_auc,
            "weighted_spearman": w_spear,
            "cls": cls,
            "mean_score": float(np.nanmean(score_fam)),
            "median_score": float(np.nanmedian(score_fam)),
        })

    return pd.DataFrame(rows)


# =========================================
# 2. Prepare base scores and corrections
# =========================================
base_candidates = {
    "base_selected_embedding": rank01(diag["base_selected_embedding_score"].values),
    "base_structure_aware": rank01(diag["base_structure_aware_score"].values),
}

retro_pen = diag["retroviral_fp_penalty"].fillna(0).values.astype(float)
ltr_corr = diag["ltr_active_correction"].fillna(0).values.astype(float)
retron_corr = diag["retron_active_rescue"].fillna(0).values.astype(float)

print("\n" + "=" * 80)
print("BASE CANDIDATES BEFORE NESTED VALIDATION")
print("=" * 80)
display(pd.DataFrame([evaluate_score_local(k, v) for k, v in base_candidates.items()]))


# =========================================
# 3. Nested LOFO validation
# =========================================
outer_logo = LeaveOneGroupOut()

a_values = np.round(np.arange(0.00, 0.151, 0.01), 3)
b_values = np.round(np.arange(0.00, 0.151, 0.01), 3)
c_values = np.round(np.arange(0.00, 0.151, 0.01), 3)

nested_oof_raw = np.zeros(len(diag))
outer_rows = []
inner_top_rows = []

for fold, (inner_idx, outer_idx) in enumerate(
    outer_logo.split(diag, y, groups=groups),
    start=1
):
    outer_family = groups[outer_idx][0]
    fold_candidates = []

    for base_name, full_base_score in base_candidates.items():

        # Inner selection uses only the non-held-out families.
        base_inner = rank01(full_base_score[inner_idx])
        base_outer = full_base_score[outer_idx]

        retro_inner = retro_pen[inner_idx]
        ltr_inner = ltr_corr[inner_idx]
        retron_inner = retron_corr[inner_idx]

        retro_outer = retro_pen[outer_idx]
        ltr_outer = ltr_corr[outer_idx]
        retron_outer = retron_corr[outer_idx]

        for a in a_values:
            for b in b_values:
                for c in c_values:
                    inner_raw = (
                        base_inner
                        - a * retro_inner
                        + b * ltr_inner
                        + c * retron_inner
                    )

                    inner_score = rank01(inner_raw)

                    inner_cls, inner_pr_auc, inner_w_spear = compute_cls(
                        y[inner_idx],
                        inner_score,
                        eff[inner_idx],
                    )

                    outer_raw = (
                        base_outer
                        - a * retro_outer
                        + b * ltr_outer
                        + c * retron_outer
                    )

                    fold_candidates.append({
                        "fold": fold,
                        "outer_family": outer_family,
                        "base_name": base_name,
                        "a_retroviral_fp_penalty": a,
                        "b_ltr_active_correction": b,
                        "c_retron_active_rescue": c,
                        "inner_pr_auc": inner_pr_auc,
                        "inner_weighted_spearman": inner_w_spear,
                        "inner_cls": inner_cls,
                        "outer_raw_score": outer_raw,
                    })

    fold_df = pd.DataFrame(fold_candidates)

    best = (
        fold_df
        .sort_values("inner_cls", ascending=False)
        .reset_index(drop=True)
        .iloc[0]
    )

    # Store raw outer score.
    # Do NOT rank the held-out family internally.
    best_outer_raw = np.asarray(best["outer_raw_score"], dtype=float)
    nested_oof_raw[outer_idx] = best_outer_raw

    # Diagnostic only: fold-internal metric for held-out family.
    if len(np.unique(y[outer_idx])) > 1:
        outer_score_diag = rank01(best_outer_raw)
        outer_cls, outer_pr_auc, outer_w_spear = compute_cls(
            y[outer_idx],
            outer_score_diag,
            eff[outer_idx],
        )
    else:
        outer_cls, outer_pr_auc, outer_w_spear = np.nan, np.nan, np.nan

    outer_rows.append({
        "fold": fold,
        "outer_family": outer_family,
        "n_outer": len(outer_idx),
        "n_active_outer": int(y[outer_idx].sum()),
        "selected_base_name": best["base_name"],
        "selected_a_retroviral_fp_penalty": best["a_retroviral_fp_penalty"],
        "selected_b_ltr_active_correction": best["b_ltr_active_correction"],
        "selected_c_retron_active_rescue": best["c_retron_active_rescue"],
        "inner_pr_auc": best["inner_pr_auc"],
        "inner_weighted_spearman": best["inner_weighted_spearman"],
        "inner_cls": best["inner_cls"],
        "outer_pr_auc_for_diagnostic_only": outer_pr_auc,
        "outer_weighted_spearman_for_diagnostic_only": outer_w_spear,
        "outer_cls_for_diagnostic_only": outer_cls,
        "outer_mean_raw_score": float(np.nanmean(best_outer_raw)),
    })

    inner_top = fold_df.drop(columns=["outer_raw_score"], errors="ignore")
    inner_top = inner_top.sort_values("inner_cls", ascending=False).head(20)
    inner_top_rows.append(inner_top)

outer_df = pd.DataFrame(outer_rows)
inner_top_df = pd.concat(inner_top_rows, ignore_index=True)

# Final nested OOF score:
# Rank all 57 raw OOF scores once.
nested_targeted_score = rank01(nested_oof_raw)

nested_cls, nested_pr_auc, nested_w_spear = compute_cls(
    y,
    nested_targeted_score,
    eff,
)

nested_summary = pd.DataFrame([
    {
        "model": "nested_targeted_correction",
        "pr_auc": nested_pr_auc,
        "weighted_spearman": nested_w_spear,
        "cls": nested_cls,
    }
])

print("\n" + "=" * 80)
print("NESTED TARGETED CORRECTION SUMMARY")
print("=" * 80)
display(nested_summary)

print("\n" + "=" * 80)
print("OUTER FOLD SETTINGS")
print("=" * 80)
display(outer_df)


# =========================================
# 4. Compare with base and full OOF targeted score
# =========================================
comparison_rows = []

for base_name, base_score in base_candidates.items():
    comparison_rows.append(evaluate_score_local(base_name, base_score))

if "targeted_correction_score" in diag.columns:
    comparison_rows.append(
        evaluate_score_local(
            "full_oof_targeted_correction_from_step_4_3",
            rank01(diag["targeted_correction_score"].values),
        )
    )

comparison_rows.append(
    evaluate_score_local(
        "nested_targeted_correction",
        nested_targeted_score,
    )
)

nested_comparison_df = (
    pd.DataFrame(comparison_rows)
    .sort_values("cls", ascending=False)
    .reset_index(drop=True)
)

print("\n" + "=" * 80)
print("MODEL COMPARISON")
print("=" * 80)
display(nested_comparison_df)


# =========================================
# 5. Family-level comparison
# =========================================
family_tables = []

for base_name, base_score in base_candidates.items():
    family_tables.append(evaluate_by_family_local(base_score, base_name))

if "targeted_correction_score" in diag.columns:
    family_tables.append(
        evaluate_by_family_local(
            rank01(diag["targeted_correction_score"].values),
            "full_oof_targeted_correction_from_step_4_3",
        )
    )

family_tables.append(
    evaluate_by_family_local(
        nested_targeted_score,
        "nested_targeted_correction",
    )
)

nested_family_compare = pd.concat(family_tables, ignore_index=True)

print("\n" + "=" * 80)
print("FAMILY-LEVEL COMPARISON")
print("=" * 80)
display(nested_family_compare.sort_values(["family", "model"]))


# =========================================
# 6. Diagnostics
# =========================================
diag["nested_targeted_correction_raw"] = nested_oof_raw
diag["nested_targeted_correction_score"] = nested_targeted_score

nested_diag = diag.copy()

print("\n" + "=" * 80)
print("TOP 25 BY NESTED TARGETED CORRECTION")
print("=" * 80)
display(
    nested_diag.sort_values("nested_targeted_correction_score", ascending=False)
    [[
        id_col, family_col, target_col, eff_col,
        "nested_targeted_correction_score",
        "base_structure_aware_score",
        "retroviral_fp_penalty",
        "ltr_active_correction",
        "retron_active_rescue",
    ]]
    .head(25)
)

print("\n" + "=" * 80)
print("TOP FALSE POSITIVES")
print("=" * 80)
display(
    nested_diag[nested_diag[target_col] == 0]
    .sort_values("nested_targeted_correction_score", ascending=False)
    [[
        id_col, family_col, target_col, eff_col,
        "nested_targeted_correction_score",
        "base_structure_aware_score",
        "retroviral_fp_penalty",
        "ltr_active_correction",
        "retron_active_rescue",
    ]]
    .head(20)
)

print("\n" + "=" * 80)
print("BOTTOM FALSE NEGATIVES")
print("=" * 80)
display(
    nested_diag[nested_diag[target_col] == 1]
    .sort_values("nested_targeted_correction_score", ascending=True)
    [[
        id_col, family_col, target_col, eff_col,
        "nested_targeted_correction_score",
        "base_structure_aware_score",
        "retroviral_fp_penalty",
        "ltr_active_correction",
        "retron_active_rescue",
    ]]
    .head(20)
)


# =========================================
# 7. Final submission only
# =========================================
submission = nested_diag[[id_col, "nested_targeted_correction_score"]].copy()
submission = submission.rename(
    columns={"nested_targeted_correction_score": "predicted_score"}
)

submission.to_csv("submission.csv", index=False)

print("\nSaved final file:")
print("- submission.csv")

print("\nSubmission preview:")
display(submission.head(20))

print("\nFinal nested targeted correction score:")
display(nested_summary)

NESTED VALIDATION INPUT
diag: (57, 117)
required columns OK

BASE CANDIDATES BEFORE NESTED VALIDATION


,model,pr_auc,weighted_spearman,cls
0,base_selected_embedding,0.658568,0.682913,0.670520
1,base_structure_aware,0.665054,0.683414,0.674109



NESTED TARGETED CORRECTION SUMMARY


,model,pr_auc,weighted_spearman,cls
0,nested_targeted_correction,0.690819,0.683706,0.687244



OUTER FOLD SETTINGS


...

,rt_name,rt_family,active,pe_efficiency_pct,nested_targeted_correction_score,base_structure_aware_score,retroviral_fp_penalty,ltr_active_correction,retron_active_rescue
29,Ne144-RT,Retron,1,0.5,0.035714,0.035714,0.039401,0.011538,0.238872
31,Rs-RT,Retron,1,2.5,0.089286,0.071429,0.003117,0.079412,0.697153
26,Ec48-RT,Retron,1,1.5,0.125000,0.107143,0.017626,0.001374,0.540272
35,Vc95-RT,Retron,1,1.5,0.267857,0.250000,0.023116,0.022945,0.795123
48,MMTV-RT,Retroviral,1,19.0,0.446429,0.446429,0.355857,0.439734,0.023143
53,SRV2-RT,Retroviral,1,3.5,0.464286,0.464286,0.419069,0.553898,0.001333
36,Vp96-RT,Retron,1,8.0,0.482143,0.500000,0.007911,0.013591,0.897629
49,MPMV-RT,Retroviral,1,7.0,0.535714,0.517857,0.361501,0.511728,0.016016
5,Er-RT,Group_II_Intron,1,12.5,0.589286,0.571429,0.000427,0.046695,0.503984
7,Gs-RT,Group_II_Intron,1,1.0,0.625000,0.625000,0.002291,0.031596,0.224621



Saved final file:
- submission.csv

Submission preview:


,rt_name,predicted_score
0,A.platensis-Cas1-RT,0.517857
1,CRISPRRT2-RT,0.357143
2,FuRT-Cas1-RT,0.607143
3,Med-CasRT,0.375000
4,ROSE-CRISPR-RT,0.285714
5,Er-RT,0.589286
6,GroupII_CLA-RT,0.178571
7,Gs-RT,0.625000
8,LtrA-RT,0.232143
9,Tel4c-RT,0.250000



Final nested targeted correction score:


,model,pr_auc,weighted_spearman,cls
0,nested_targeted_correction,0.690819,0.683706,0.687244
